# Sección 0 — Imports

In [55]:
import datetime, glob, os, re
import numpy as np
import pandas as pd
import openpyxl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from ipywidgets import interact, Dropdown, FloatSlider, HBox, VBox, Output, BoundedIntText
import ipywidgets as widgets
from IPython.display import display, HTML

# Sección 0.5 — Globals

In [56]:
# ── Constantes ───────────────────────────────────────────────────────────────
CUARTO_AMANECER  = 600
CUARTO_ATARDECER = 1815
RAMP_BIN_MW      = 10
BIN_SIZE         = 0.5

tec_color = {
    'Hidro':   '#378ADD',
    'Solar':   '#E8A838',
    'Eolica':  '#1D9E75',
    'Termica': '#D85A30',
}

MESES_NOMBRES = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril',
                 5:'Mayo',  6:'Junio',   7:'Julio', 8:'Agosto',
                 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}

PLOT_BG    = '#141414'
PAPER_BG   = '#1a1a1a'
FONT_COLOR = '#cccccc'
GRID_COLOR = 'rgba(255,255,255,0.06)'

SEV_CRITICO  = ('CRITICO',  '#e74c3c')
SEV_ATENCION = ('ATENCION', '#f39c12')
SEV_OK       = ('OK',       '#2ecc71')

STATS_COLS = ['MINIMA', 'MODA_BAJA', 'MEDIANA', 'MODA_ALTA', 'MAXIMA']

CUARTO_ESTRES_1_INI = 1000
CUARTO_ESTRES_1_FIN = 1400
CUARTO_ESTRES_2_INI = 1400
CUARTO_ESTRES_2_FIN = 1700

def hex_to_rgba(hex_color, opacity=0.18):
    h = hex_color.lstrip('#')
    r,g,b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{opacity})'

# ── Capacidad instalada por planta (MW) ───────────────────────────────────────
inst_mw_manual = {
    'BAYANO':260.0,'BLM':0.0,'TROPITERMICA':5.05,'TERMOCOLON':150.0,
    'PANAM':99.80,'MENDOZA SOLAR':3.0,'SUNRISE MASPV1':0.50,'SUNRISE MASPV2':3.30,
    'CHAME SOLAR':20.0,'CAPIRA SOLAR':9.9,'SPARKLE 1':15.30,'SPARKLE 2':34.80,
    'SAN CARLOS SOLAR':9.99,'RODEO SOLAR':9.9,'PROYECTO GATUN':670.0,'CATIVA':87.0,
    'MIRAFLORES':99.62,'PACORA':53.53,'PACORA II':3.0,'COSTA NORTE GAS':381.0,
    'URBALIA CERRO PATACON':8.10,'BARCAZA ESPERANZA':50.0,'JINRO POWER':50.0,
    'PANAM II':49.80,'COBRE PANAMA':300.0,'PARQUE EOLICO TOABRE':66.0,
    'NUEVO CHAGRES 1':55.0,'NUEVO CHAGRES 2':62.5,'ROSA VIENTOS 1':52.5,
    'ROSA VIENTOS 2':50.0,'MARANON':17.5,'PORTOBELO':32.5,
    'SOLAR FOTOVOLTAICA PENONOME':120.0,'LA YEGUADA':7.0,'EL FRAILE':6.66,
    'DIVISA SOLAR':9.99,'DON FELIX':9.99,'SOLAR PARIS':8.99,'FARALLON SOLAR 2':5.16,
    'SOLAR LOS ANGELES':9.52,'ESTRELLA SOLAR':4.93,'MILTON SOLAR':10.0,
    'VISTA ALEGRE':8.22,'SOL REAL':10.00,'EL ESPINAL':8.5,'SOLAR POCRI':16.0,
    'BEJUCO SOLAR':0.96,'PANASOLAR':9.90,'EL FRAILE SOLAR 1':0.48,'PESE SOLAR':9.98,
    'MAYORCA SOLAR':9.97,'JAGUITO SOLAR':9.99,'ORO SOLAR':5.00,
    'LOS SANTOS SOLAR':7.56,'RIO DE JESUS SOLAR':5.0,'LA VILLA SOLAR':9.99,
    'COROTU SOLAR':9.98,'SARIGUA':2.40,'SOLAR COCLE':8.99,'LAS CRUCES':18.83,
    'MINI LAS CRUCES':0.97,'DACONAN STAR SOLAR':3.24,'PANASOLAR II':5.0,
    'PANASOLAR III':5.0,'ANTON SOLAR 1':9.99,'ECOENER SAN JUAN':5.0,
    'CAMPO SOLAR SANTIAGO 1':10.0,'CAMPO SOLAR SANTIAGO 2':10.0,
    'CAMPO SOLAR SANTIAGO 3':10.0,'CAMPO SOLAR SANTIAGO 4':10.0,
    'CAMPO SOLAR SANTIAGO 5':10.0,'CAMPO SOLAR SANTIAGO 6':10.0,
    'CAMPO SOLAR SANTIAGO 7':10.0,'FOTOVOLTAICA SANTIAGO GEN 1':5.0,
    'LA ESPERANZA SOLAR 20 MW':19.99,'MADRE VIEJA SOLAR':26.0,'CEDRO SOLAR':9.96,
    'CAOBA SOLAR':9.96,'MACANO SOLAR':4.75,'PEDREGALITO SOLAR POWER':10.0,
    'CONCEPCION':10.0,'MACANO':5.80,'PEDREGALITO 2':12.52,'LAS PERLAS NORTE':10.0,
    'LAS PERLAS SUR':10.0,'RP490':14.00,'LA CUCHILLA':7.62,'PEDREGALITO 1':19.86,
    'CADASA':30.0,'MINI BARRO BLANCO':1.89,'BARRO BLANCO':26.95,'LOS VALLES':54.8,
    'MENDRE':19.75,'MENDRE 2':8.0,'ALGARROBOS':9.73,'COCHEA':15.50,
    'LA ESTRELLA':47.2,'MENDRE SOLAR':5.5,'BONYIC':31.8,'MONTE LIRIO':51.65,
    'PANDO':32.6,'EL ALTO':72.20,'MINI CENTRAL EL ALTO':1.14,'CHANGUINOLA 1':213.2,
    'MINI CHAN':9.78,'FORTUNA':300.0,'GUALACA':25.30,'LORENA':35.0,
    'PRUDENCIA':58.69,'ESTI':120.0,'PARQUE SOLAR PRUDENCIA':9.69,'SAN LORENZO':8.12,
    'DOLEGA':3.12,'MACHO MONTE':2.5,'PASO ANCHO':6.00,'LOS PLANETAS 1':4.75,
    'LOS PLANETAS 2':8.89,'BUGABA 1':5.12,'BUGABA 2':5.86,'SAN ANDRES':10.0,
    'BAJOS DEL TOTUMA':6.30,'SOLAR CHIRIQUI':9.87,'SOLAR BUGABA':2.56,
    'IKAKO':10.0,'ANDREAS POWER':0.99,'SALSIPUEDES':27.9,'LA POTRA':27.9,
    'BAJO DE MINA':56.80,'BAITUN':85.90,'MINI BAJOMINA':0.60,'MINI LA POTRA':2.10,
    'MINI BAITUN':1.70,'SOL DE DAVID':7.90,'SOLAR CALDERA':5.45,'ECOSOLAR':10.0,
    'SOLARPRO':10.0,'BACO SOLAR':25.90,'ECO HIDRO TIZINGAL':0.0,
    'MINI PEDREGALITO 1':2.5,'INTERCAMBIO':0.0,'ZONA FRANCA ALBROOK':0.1,
    'CHUPAMPA SOLAR':9.9,
}

print(f"Globals cargados.")

Globals cargados.


In [57]:
# ── Estaciones ────────────────────────────────────────────────────────────────
MESES_VERANO   = {12, 1, 2, 3, 4}      # Dic – Abr
MESES_LLUVIOSO = {5, 6, 7, 8, 9, 10, 11} # May – Nov

# ── Estilos por estadístico ───────────────────────────────────────────────────
STAT_EST = {
    'MINIMA':    dict(dash='dot',     width=1.5, opacity=0.50),
    'MODA_BAJA': dict(dash='dashdot', width=1.8, opacity=0.70),
    'MEDIANA':   dict(dash='dash',    width=1.8, opacity=0.65),
    'MODA_ALTA': dict(dash='solid',   width=2.5, opacity=1.00),
    'MAXIMA':    dict(dash='dot',     width=1.5, opacity=0.45),
}

# ── Layout oscuro sin titulo ──────────────────────────────────────────────────
def _layout_oscuro(fig, title='', height=520, y_title='MW'):
    """title aceptado por compatibilidad pero no mostrado."""
    fig.update_layout(
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        yaxis=dict(title=y_title, showgrid=True,
                   gridcolor='rgba(255,255,255,0.06)', zeroline=False),
        xaxis=dict(showgrid=True,
                   gridcolor='rgba(255,255,255,0.06)', zeroline=False),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)',
                    font=dict(size=11)),
        hovermode='x unified',
        height=height,
        margin=dict(t=40, l=65, r=40, b=60),
    )

print("STAT_EST y _layout_oscuro cargados.")

STAT_EST y _layout_oscuro cargados.


# Sección 1 — Parámetros y carga

In [58]:
# ── Parámetros editables ──────────────────────────────────────────────────────
CARPETA_STATS = '.'
AÑO_ANALISIS  = 2025
MES_ANALISIS  = [1,2,3,4,5,6,7,8,9,10,11,12]   # int, lista o 'todos'

In [59]:
import glob as _glob

def _resolver_rutas(carpeta, año, mes_analisis):
    patron_todos = sorted(_glob.glob(
        os.path.join(carpeta, f'STATS_WIDE_{año}_*.xlsx')
    ))
    if not patron_todos:
        raise FileNotFoundError(
            f'No se encontro ningun STATS_WIDE_{año}_*.xlsx en '
            f'{os.path.abspath(carpeta)}'
        )
    if mes_analisis == 'todos':
        return patron_todos
    meses = [mes_analisis] if isinstance(mes_analisis, int) else list(mes_analisis)
    mes_str_buscado = '_'.join(f'{m:02d}' for m in sorted(meses))
    combinado = [r for r in patron_todos if mes_str_buscado in os.path.basename(r)]
    if combinado:
        return sorted(combinado)
    encontrados = []
    for m in meses:
        parciales = [r for r in patron_todos if f'_{m:02d}' in os.path.basename(r)]
        encontrados.extend(parciales)
    if not encontrados:
        raise FileNotFoundError(
            f'No se encontraron archivos para meses {meses} en {os.path.abspath(carpeta)}'
        )
    return sorted(set(encontrados))

RUTAS_STATS_WIDE = _resolver_rutas(CARPETA_STATS, AÑO_ANALISIS, MES_ANALISIS)
print(f"Archivos a cargar: {[os.path.basename(r) for r in RUTAS_STATS_WIDE]}")

Archivos a cargar: ['STATS_WIDE_2025_01_02_03_04_05_06_07_08_09_10_11_12.xlsx']


In [60]:
# ── Mapeo interno ─────────────────────────────────────────────────────────────
# Nombres de columna internos = sufijos exactos de sheet en el Excel.
# Sheets: NIVEL_MINIMA, NIVEL_MODA_BAJA, NIVEL_MEDIANA, NIVEL_MODA_ALTA, NIVEL_MAXIMA
# Rampas: NIVEL_RAMP_UP_MAX, NIVEL_RAMP_DOWN_MAX
_STAT_MAP = [
    ('MINIMA',    'MINIMA'),
    ('MODA_BAJA', 'MODA_BAJA'),
    ('MEDIANA',   'MEDIANA'),
    ('MODA_ALTA', 'MODA_ALTA'),
    ('MAXIMA',    'MAXIMA'),
]

# Etiqueta para nodos cuyo nombre viene en blanco en el Excel (antes se perdian).
NODO_SIN_NOMBRE = 'SIN_NODO'


def _localizar_estructura(raw, n_meta):
    """
    Deriva las filas de cabecera de forma robusta, sin asumir offsets fijos.
    La fila de etiquetas es la que contiene 'CAPACIDAD_INSTALADA_MW' en col n_meta-1.
    Estructura observada y validada en las 14 hojas:
        fila_mes    = label - 2   (numeros de MES)
        fila_cuarto = label - 1   (CUARTO en stats / DIA en rampas)
        fila_datos  = label + 1
    Devuelve (fila_mes, fila_cuarto, fila_label, fila_datos).
    """
    col_cap = n_meta - 1
    fila_label = None
    for i in range(min(6, raw.shape[0])):
        val = raw.iloc[i, col_cap]
        if isinstance(val, str) and 'CAPACIDAD' in val.upper():
            fila_label = i
            break
    if fila_label is None:
        fila_label = 2  # respaldo: estructura estandar observada
    return fila_label - 2, fila_label - 1, fila_label, fila_label + 1


def _leer_nivel(rutas, nivel, meta_cols):
    """
    Lector universal para TECNOLOGIA, NODO o PLANTAS.

    Parametros
    ----------
    rutas      : list[str]  rutas a archivos STATS_WIDE_*.xlsx
    nivel      : str        'TECNOLOGIA', 'NODO' o 'PLANTAS'
    meta_cols  : list[str]  primeras columnas de cada hoja
                            ['TECNOLOGIA','CAP_MW'] o ['NODO','TECNOLOGIA','CAP_MW']

    Devuelve
    --------
    df_stats : DataFrame  columnas = meta_cols + STATS_COLS + ['MES','CUARTO']
    df_ramp  : DataFrame  columnas = meta_cols + ['MES','DIA','RAMP_UP_MAX','RAMP_DOWN_MAX']
               (vacio si el Excel no tiene hojas de rampas para el nivel)

    Notas
    -----
    Cada estadistico y cada direccion de rampa se leen de su hoja dedicada.
    Las hojas de rampas son opcionales: se detectan primero las hojas presentes
    en el Excel y solo se leen si existen (PLANTAS no las tiene).
    Los valores se conservan crudos (MW); no se normalizan por capacidad aqui.
    """
    if isinstance(rutas, str):
        rutas = [rutas]
    n_meta   = len(meta_cols)
    stat_int = [s for s, _ in _STAT_MAP]
    # La columna TECNOLOGIA esta presente en ambos niveles y discrimina las filas
    # de datos reales (col 0 en TECNOLOGIA, col 1 en NODO). Filtrar por ella evita
    # descartar la fila cuyo NODO viene en blanco (bug del lector v8).
    tec_col  = meta_cols.index('TECNOLOGIA')

    def _filtrar_y_etiquetar(d):
        tname = meta_cols[tec_col]
        d = d[d[tname].notna() & (d[tname].astype(str).str.strip() != '')]
        if nivel == 'NODO':
            blank = d[meta_cols[0]].isna() | (d[meta_cols[0]].astype(str).str.strip() == '')
            if blank.any():
                d = d.copy()
                d.loc[blank, meta_cols[0]] = NODO_SIN_NOMBRE
        return d

    maestros, ramp_bloqs = [], []
    for ruta in rutas:
        _hojas_excel = pd.ExcelFile(ruta).sheet_names
        # ── Estadisticos ─────────────────────────────────────────────────────
        dfs_melted = {}
        for col_name, sheet_suf in _STAT_MAP:
            hoja = f'{nivel}_{sheet_suf}'
            raw  = pd.read_excel(ruta, sheet_name=hoja, header=None)
            f_mes, f_cuarto, f_label, f_dato = _localizar_estructura(raw, n_meta)
            if raw.shape[0] <= f_dato or raw.shape[1] <= n_meta:
                raise ValueError(f'Hoja {hoja} forma inesperada: {raw.shape}')
            data = raw.iloc[f_dato:].copy().reset_index(drop=True)
            n    = raw.shape[1]
            data.columns = meta_cols + [f'__v{j}' for j in range(n_meta, n)]
            data = _filtrar_y_etiquetar(data)
            data['CAP_MW'] = pd.to_numeric(data['CAP_MW'], errors='coerce').fillna(0.0)
            val_cols   = [f'__v{j}' for j in range(n_meta, n)]
            mes_map    = {f'__v{j}': raw.iloc[f_mes, j]    for j in range(n_meta, n)}
            cuarto_map = {f'__v{j}': raw.iloc[f_cuarto, j] for j in range(n_meta, n)}
            long_df = data.melt(id_vars=meta_cols, value_vars=val_cols,
                                var_name='_colid', value_name=col_name)
            long_df['MES']    = pd.to_numeric(long_df['_colid'].map(mes_map),    errors='coerce')
            long_df['CUARTO'] = pd.to_numeric(long_df['_colid'].map(cuarto_map), errors='coerce')
            dfs_melted[col_name] = long_df.drop(columns='_colid')

        keys = meta_cols + ['MES', 'CUARTO']
        df   = dfs_melted[stat_int[0]]
        for s in stat_int[1:]:
            df = df.merge(dfs_melted[s], on=keys, how='outer')
        df['MES']    = pd.to_numeric(df['MES'],    errors='coerce')
        df['CUARTO'] = pd.to_numeric(df['CUARTO'], errors='coerce')
        df = df.dropna(subset=['MES', 'CUARTO'])
        df['MES']    = df['MES'].astype(int)
        df['CUARTO'] = df['CUARTO'].astype(int)
        df = df[df[meta_cols[0]].notna()]
        for s in stat_int:
            df[s] = pd.to_numeric(df[s], errors='coerce').fillna(0.0)
        maestros.append(df)

        # ── Rampas (solo si las hojas existen en el Excel) ───────────────────
        for ramp_col, ramp_suf in [('RAMP_UP_MAX',   'RAMP_UP_MAX'),
                                    ('RAMP_DOWN_MAX', 'RAMP_DOWN_MAX')]:
            hoja_r  = f'{nivel}_{ramp_suf}'
            if hoja_r not in _hojas_excel:
                continue
            raw_r   = pd.read_excel(ruta, sheet_name=hoja_r, header=None)
            f_mes, f_dia, f_label, f_dato = _localizar_estructura(raw_r, n_meta)
            data_r  = raw_r.iloc[f_dato:].copy().reset_index(drop=True)
            n_r     = raw_r.shape[1]
            data_r.columns = meta_cols + [f'__r{j}' for j in range(n_meta, n_r)]
            data_r  = _filtrar_y_etiquetar(data_r)
            val_r   = [f'__r{j}' for j in range(n_meta, n_r)]
            mes_r   = {f'__r{j}': raw_r.iloc[f_mes, j] for j in range(n_meta, n_r)}
            dia_r   = {f'__r{j}': raw_r.iloc[f_dia, j] for j in range(n_meta, n_r)}
            long_r  = data_r.melt(id_vars=meta_cols, value_vars=val_r,
                                  var_name='_colid', value_name=ramp_col)
            long_r['MES'] = pd.to_numeric(long_r['_colid'].map(mes_r), errors='coerce')
            long_r['DIA'] = pd.to_numeric(long_r['_colid'].map(dia_r), errors='coerce')
            long_r = long_r.drop(columns='_colid').dropna(subset=['MES', 'DIA'])
            long_r['MES'] = long_r['MES'].astype(int)
            long_r['DIA'] = long_r['DIA'].astype(int)
            long_r[ramp_col] = pd.to_numeric(long_r[ramp_col], errors='coerce').fillna(0.0)
            ramp_bloqs.append(long_r[meta_cols + ['MES', 'DIA', ramp_col]])

    # ── Combinar rutas ────────────────────────────────────────────────────────
    df_final = pd.concat(maestros, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=meta_cols + ['MES', 'CUARTO'])

    if ramp_bloqs:
        df_ramp = ramp_bloqs[0]
        for blq in ramp_bloqs[1:]:
            df_ramp = df_ramp.merge(blq, on=meta_cols + ['MES', 'DIA'], how='outer')
        df_ramp = df_ramp.drop_duplicates(subset=meta_cols + ['MES', 'DIA'])
        for rc in ['RAMP_UP_MAX', 'RAMP_DOWN_MAX']:
            if rc not in df_ramp.columns:
                df_ramp[rc] = 0.0
            df_ramp[rc] = pd.to_numeric(df_ramp[rc], errors='coerce').fillna(0.0)
    else:
        df_ramp = pd.DataFrame(
            columns=meta_cols + ['MES', 'DIA', 'RAMP_UP_MAX', 'RAMP_DOWN_MAX'])

    print(f"[{nivel}] stats={df_final.shape}  "
          f"meses={sorted(df_final['MES'].unique().astype(int).tolist())}  "
          f"ramp={df_ramp.shape}")
    _validar_capacidad(df_final, meta_cols, nivel)
    return df_final, df_ramp


def _validar_capacidad(df, meta_cols, nivel):
    """
    Reporte de calidad de datos (NO modifica valores).
    Marca series donde la generacion supera la capacidad registrada (factor>100%)
    y entidades con capacidad registrada = 0 pero con generacion. Estos valores son
    REALES en la fuente (p.ej. capacidad agregada a mitad de ano o cap subregistrada)
    y se conservan tal cual; el reporte solo los hace visibles.
    """
    ent = [c for c in meta_cols if c != 'CAP_MW']
    g = (df.groupby(ent, dropna=False)
           .agg(CAP_MW=('CAP_MW', 'first'),
                MODA_ALTA=('MODA_ALTA', 'max'),
                MAXIMA=('MAXIMA', 'max')).reset_index())
    over = g[(g['CAP_MW'] > 0) & (g['MODA_ALTA'] > g['CAP_MW'])].copy()
    over['FACTOR'] = (over['MODA_ALTA'] / over['CAP_MW'] * 100).round(1)
    cap0 = g[(g['CAP_MW'] <= 0) & (g['MAXIMA'] > 0)]
    print(f"  [VALIDACION {nivel}] series factor>100%: {len(over)} | "
          f"cap=0 con generacion: {len(cap0)}")
    for _, r in over.sort_values('FACTOR', ascending=False).iterrows():
        etq = ' · '.join(str(r[c]) for c in ent)
        print(f"     FLAG factor>100%  {etq:24s} cap={r['CAP_MW']:8.2f}  "
              f"MODA_ALTA={r['MODA_ALTA']:8.2f}  ({r['FACTOR']:.1f}%)")
    for _, r in cap0.iterrows():
        etq = ' · '.join(str(r[c]) for c in ent)
        print(f"     FLAG cap=0        {etq:24s} cap=0.00  "
              f"MAXIMA={r['MAXIMA']:8.2f}  (factor indefinido)")
    return over, cap0




def _leer_gen_solar_ramp(rutas):
    """
    Lee las hojas opcionales TECNOLOGIA_RAMP_UP_GEN_SOLAR y
    TECNOLOGIA_RAMP_DOWN_GEN_SOLAR del STATS_WIDE.

    Devuelve (df_gen_up, df_gen_down) con columnas
    [TECNOLOGIA, MES, DIA, GEN_SOLAR].

    Si las hojas no existen (STATS_WIDE generado antes del cambio en
    MENSUAL_DEP) devuelve DataFrames vacios sin lanzar error.
    """
    if isinstance(rutas, str):
        rutas = [rutas]

    _META  = ['TECNOLOGIA', 'CAP_MW']
    _HOJAS = {
        'up':   'TECNOLOGIA_RAMP_UP_GEN_SOLAR',
        'down': 'TECNOLOGIA_RAMP_DOWN_GEN_SOLAR',
    }
    bloqs = {'up': [], 'down': []}

    for ruta in rutas:
        try:
            xl = pd.ExcelFile(ruta)
        except Exception as e:
            print(f"   [GEN_SOLAR] No se pudo abrir {os.path.basename(ruta)}: {e}")
            continue

        for direccion, hoja in _HOJAS.items():
            if hoja not in xl.sheet_names:
                continue
            try:
                raw    = pd.read_excel(ruta, sheet_name=hoja, header=None)
                n_meta = len(_META)
                f_mes, f_dia, f_label, f_dato = _localizar_estructura(raw, n_meta)
                data   = raw.iloc[f_dato:].copy().reset_index(drop=True)
                n      = raw.shape[1]
                data.columns = _META + [f'__v{j}' for j in range(n_meta, n)]
                data = data[
                    data['TECNOLOGIA'].notna() &
                    (data['TECNOLOGIA'].astype(str).str.strip() != '')
                ]
                val_cols = [f'__v{j}' for j in range(n_meta, n)]
                mes_map  = {f'__v{j}': raw.iloc[f_mes, j]  for j in range(n_meta, n)}
                dia_map  = {f'__v{j}': raw.iloc[f_dia, j]  for j in range(n_meta, n)}
                long_df  = data.melt(
                    id_vars=_META, value_vars=val_cols,
                    var_name='_colid', value_name='GEN_SOLAR'
                )
                long_df['MES'] = pd.to_numeric(long_df['_colid'].map(mes_map), errors='coerce')
                long_df['DIA'] = pd.to_numeric(long_df['_colid'].map(dia_map), errors='coerce')
                long_df = (long_df
                           .drop(columns='_colid')
                           .dropna(subset=['MES', 'DIA']))
                long_df['MES']       = long_df['MES'].astype(int)
                long_df['DIA']       = long_df['DIA'].astype(int)
                long_df['GEN_SOLAR'] = pd.to_numeric(
                    long_df['GEN_SOLAR'], errors='coerce').fillna(0.0)
                bloqs[direccion].append(long_df[['TECNOLOGIA', 'MES', 'DIA', 'GEN_SOLAR']])
            except Exception as e:
                print(f"   [GEN_SOLAR] {hoja} en {os.path.basename(ruta)}: {e}")

    _empty = pd.DataFrame(columns=['TECNOLOGIA', 'MES', 'DIA', 'GEN_SOLAR'])
    df_gen_up = (
        pd.concat(bloqs['up'], ignore_index=True)
          .drop_duplicates(subset=['TECNOLOGIA', 'MES', 'DIA'])
        if bloqs['up'] else _empty.copy()
    )
    df_gen_down = (
        pd.concat(bloqs['down'], ignore_index=True)
          .drop_duplicates(subset=['TECNOLOGIA', 'MES', 'DIA'])
        if bloqs['down'] else _empty.copy()
    )
    print(f"[GEN_SOLAR] up={df_gen_up.shape}  down={df_gen_down.shape}")
    return df_gen_up, df_gen_down

# ── Wrappers ──────────────────────────────────────────────────────────────────
def _leer_stats_wide(rutas):
    """TECNOLOGIA: 2 meta cols [TECNOLOGIA, CAP_MW]."""
    return _leer_nivel(rutas, 'TECNOLOGIA', ['TECNOLOGIA', 'CAP_MW'])

def _leer_stats_nodo(rutas):
    """NODO: 3 meta cols [NODO, TECNOLOGIA, CAP_MW]."""
    return _leer_nivel(rutas, 'NODO', ['NODO', 'TECNOLOGIA', 'CAP_MW'])

def _leer_stats_planta(rutas):
    """PLANTAS: 5 meta cols [PLANTA, TECNOLOGIA, NODO, ZONA, CAP_MW]."""
    return _leer_nivel(rutas, 'PLANTAS', ['PLANTA', 'TECNOLOGIA', 'NODO', 'ZONA', 'CAP_MW'])


# ── Carga ─────────────────────────────────────────────────────────────────────
df_maestro, df_ramp_tec = _leer_stats_wide(RUTAS_STATS_WIDE)


[TECNOLOGIA] stats=(4608, 9)  meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]  ramp=(1488, 6)
  [VALIDACION TECNOLOGIA] series factor>100%: 0 | cap=0 con generacion: 0


In [61]:
df_maestro_nodo, df_ramp_nodo = _leer_stats_nodo(RUTAS_STATS_WIDE)
df_maestro_planta, _ = _leer_stats_planta(RUTAS_STATS_WIDE)
df_gen_solar_up, df_gen_solar_down = _leer_gen_solar_ramp(RUTAS_STATS_WIDE)


[NODO] stats=(43776, 10)  meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]  ramp=(14136, 7)
  [VALIDACION NODO] series factor>100%: 8 | cap=0 con generacion: 1
     FLAG factor>100%  EHI · Solar              cap=   29.88  MODA_ALTA=   50.25  (168.2%)
     FLAG factor>100%  CPA · Termica            cap=   50.10  MODA_ALTA=   60.25  (120.3%)
     FLAG factor>100%  CAL · Solar              cap=    5.50  MODA_ALTA=    5.75  (104.5%)
     FLAG factor>100%  PAC · Termica            cap=   53.53  MODA_ALTA=   55.25  (103.2%)
     FLAG factor>100%  SMA · Termica            cap=    8.10  MODA_ALTA=    8.25  (101.9%)
     FLAG factor>100%  CHA · Hidro              cap=   31.80  MODA_ALTA=   32.25  (101.4%)
     FLAG factor>100%  DOM · Hidro              cap=  157.59  MODA_ALTA=  159.75  (101.4%)
     FLAG factor>100%  GUA · Hidro              cap=  238.99  MODA_ALTA=  240.25  (100.5%)
     FLAG cap=0        BLM · Termica            cap=0.00  MAXIMA=   30.69  (factor indefinido)
[PLANTAS] stats=(188

# Sección 2 — Vistas derivadas

In [62]:
_agg = {'MINIMA':'min','MODA_BAJA':'sum','MEDIANA':'sum','MODA_ALTA':'sum','MAXIMA':'max'}

df_tec = (
    df_maestro
    .groupby(['MES','CUARTO','TECNOLOGIA'], as_index=False, dropna=False)
    .agg(_agg)
)
df_tec['TURNO'] = df_tec['CUARTO'].apply(
    lambda c: 'DIA' if CUARTO_AMANECER <= c <= CUARTO_ATARDECER else 'NOCHE'
)
df_tec['N_DIAS']     = 1
df_tec['PCT_ACTIVO'] = 100.0

_CAP_TEC = (
    df_maestro
    .drop_duplicates(subset=['TECNOLOGIA'])
    .set_index('TECNOLOGIA')['CAP_MW']
    .to_dict()
)
CAP_SOLAR_ACTUAL = float(_CAP_TEC.get('Solar', 0))
MESES_DISP       = sorted(df_maestro['MES'].unique().astype(int).tolist())
TECS_DISP        = sorted(df_maestro['TECNOLOGIA'].dropna().unique().tolist())
MESES_OPCIONES   = MESES_DISP

print(f"df_tec shape={df_tec.shape}")
print(f"CAP_SOLAR_ACTUAL = {CAP_SOLAR_ACTUAL:.1f} MW")
print(f"MESES_DISP       = {MESES_DISP}")
print(f"TECS_DISP        = {TECS_DISP}")

df_tec shape=(4608, 11)
CAP_SOLAR_ACTUAL = 860.0 MW
MESES_DISP       = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
TECS_DISP        = ['Eolica', 'Hidro', 'Solar', 'Termica']


In [63]:
# ── Datos de Nodo ─────────────────────────────────────────────────────────────
df_maestro_nodo, df_ramp_nodo = _leer_stats_nodo(RUTAS_STATS_WIDE)

df_nodo               = df_maestro_nodo.copy()
df_nodo['N_DIAS']     = 1
df_nodo['PCT_ACTIVO'] = 100.0

# Mascara solar nocturna
_sol  = df_nodo['TECNOLOGIA'] == 'Solar'
_noch = (df_nodo['CUARTO'] <= CUARTO_AMANECER) | (df_nodo['CUARTO'] >= CUARTO_ATARDECER)
for _s in STATS_COLS:
    df_nodo.loc[_sol & _noch, _s] = 0.0

NODOS_DISP    = sorted(df_nodo['NODO'].dropna().unique().tolist())
_CAP_NODO_TEC = (df_maestro_nodo.drop_duplicates(subset=['NODO','TECNOLOGIA'])
                 .set_index(['NODO','TECNOLOGIA'])['CAP_MW'].to_dict())
NODO_TECS_DISP = {n: sorted(df_nodo[df_nodo['NODO']==n]['TECNOLOGIA'].unique().tolist())
                  for n in NODOS_DISP}

print(f"Nodos cargados: {len(NODOS_DISP)}  |  Primeros 5: {NODOS_DISP[:5]}")

[NODO] stats=(43776, 10)  meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]  ramp=(14136, 7)
  [VALIDACION NODO] series factor>100%: 8 | cap=0 con generacion: 1
     FLAG factor>100%  EHI · Solar              cap=   29.88  MODA_ALTA=   50.25  (168.2%)
     FLAG factor>100%  CPA · Termica            cap=   50.10  MODA_ALTA=   60.25  (120.3%)
     FLAG factor>100%  CAL · Solar              cap=    5.50  MODA_ALTA=    5.75  (104.5%)
     FLAG factor>100%  PAC · Termica            cap=   53.53  MODA_ALTA=   55.25  (103.2%)
     FLAG factor>100%  SMA · Termica            cap=    8.10  MODA_ALTA=    8.25  (101.9%)
     FLAG factor>100%  CHA · Hidro              cap=   31.80  MODA_ALTA=   32.25  (101.4%)
     FLAG factor>100%  DOM · Hidro              cap=  157.59  MODA_ALTA=  159.75  (101.4%)
     FLAG factor>100%  GUA · Hidro              cap=  238.99  MODA_ALTA=  240.25  (100.5%)
     FLAG cap=0        BLM · Termica            cap=0.00  MAXIMA=   30.69  (factor indefinido)
Nodos cargados: 26  

In [64]:
# ── Datos de Planta ───────────────────────────────────────────────────────────
df_maestro_planta, _ = _leer_stats_planta(RUTAS_STATS_WIDE)

df_planta               = df_maestro_planta.copy()
df_planta['N_DIAS']     = 1
df_planta['PCT_ACTIVO'] = 100.0

# Mascara solar nocturna
_sol  = df_planta['TECNOLOGIA'] == 'Solar'
_noch = (df_planta['CUARTO'] <= CUARTO_AMANECER) | (df_planta['CUARTO'] >= CUARTO_ATARDECER)
for _s in STATS_COLS:
    df_planta.loc[_sol & _noch, _s] = 0.0

PLANTAS_DISP = sorted(df_planta['PLANTA'].dropna().unique().tolist())
_meta_planta = df_maestro_planta.drop_duplicates(subset=['PLANTA']).set_index('PLANTA')
_CAP_PLANTA  = _meta_planta['CAP_MW'].to_dict()
_TEC_PLANTA  = _meta_planta['TECNOLOGIA'].to_dict()
_NODO_PLANTA = _meta_planta['NODO'].to_dict()
_ZONA_PLANTA = _meta_planta['ZONA'].to_dict()

print(f"Plantas cargadas: {len(PLANTAS_DISP)}  |  Primeras 5: {PLANTAS_DISP[:5]}")

[PLANTAS] stats=(188928, 12)  meses=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]  ramp=(0, 9)
  [VALIDACION PLANTAS] series factor>100%: 54 | cap=0 con generacion: 2
     FLAG factor>100%  ESTRELLA SOLAR · Solar · LSA · Zona Central cap=    4.93  MODA_ALTA=   15.25  (309.3%)
     FLAG factor>100%  RIO DE JESUS SOLAR · Solar · LSA · Zona Central cap=    5.00  MODA_ALTA=   15.25  (305.0%)
     FLAG factor>100%  ANTON SOLAR 1 · Solar · EHI · Zona Central cap=    9.99  MODA_ALTA=   24.75  (247.7%)
     FLAG factor>100%  SPARKLE 1 · Termica · CPA · Zona Oriental cap=   15.30  MODA_ALTA=   30.25  (197.7%)
     FLAG factor>100%  ORO SOLAR · Solar · LSA · Zona Central cap=    5.00  MODA_ALTA=    9.75  (195.0%)
     FLAG factor>100%  FARALLON SOLAR 2 · Solar · LSA · Zona Central cap=    5.16  MODA_ALTA=    9.25  (179.3%)
     FLAG factor>100%  DACONAN STAR SOLAR · Solar · LSA · Zona Central cap=    3.24  MODA_ALTA=    5.75  (177.5%)
     FLAG factor>100%  EL ESPINAL · Solar · LSA · Zona Central cap=

# Sección 3 — Helpers visuales

In [65]:
def _agregar_amanecer_atardecer(fig):
    fig.add_vline(x=CUARTO_AMANECER,  line_color='rgba(232,168,56,0.45)',
                  line_dash='dot', line_width=1,
                  annotation_text='Amanecer',
                  annotation_font=dict(size=10, color='#E8A838'),
                  annotation_position='top')
    fig.add_vline(x=CUARTO_ATARDECER, line_color='rgba(232,168,56,0.45)',
                  line_dash='dot', line_width=1,
                  annotation_text='Atardecer',
                  annotation_font=dict(size=10, color='#E8A838'),
                  annotation_position='top')
    return fig

def _resumen_critico(items, titulo='Resumen'):
    if not items: return
    orden = {'CRITICO':0,'ATENCION':1,'OK':2}
    sev_dom = min((s for s,_ in items), key=lambda s: orden[s[0]])
    borde   = sev_dom[1]
    filas = ''
    for (tag, color), texto in items:
        filas += (
            f'<div style="margin:6px 0; font-size:12px;">'
            f'<span style="display:inline-block; min-width:90px; padding:2px 8px;'
            f' border-radius:3px; background:{color}; color:#0e0e0e;'
            f' font-weight:700; font-size:11px; text-align:center;">{tag}</span>'
            f' &nbsp;{texto}</div>'
        )
    html = (
        f'<div style="background:#1f1f1f; border-left:5px solid {borde};'
        f' padding:10px 14px; margin:8px 0;'
        f' color:{FONT_COLOR}; font-family:Inter, sans-serif;">'
        f'<div style="font-weight:700; font-size:13px; margin-bottom:4px;">'
        f'{titulo}</div>{filas}</div>'
    )
    display(HTML(html))

def _stats_por_cuarto(df, filtros, ordenar=True):
    sub = df.copy()
    for col, val in filtros.items():
        sub = sub[sub[col] == val]
    if sub.empty: return sub
    if sub['CUARTO'].duplicated().any():
        sub = sub.groupby('CUARTO', as_index=False).agg(
            MINIMA     =('MINIMA',     'min'),
            MODA_BAJA  =('MODA_BAJA',  'mean'),
            MEDIANA    =('MEDIANA',    'mean'),
            MODA_ALTA  =('MODA_ALTA',  'mean'),
            MAXIMA     =('MAXIMA',     'max'),
            N_DIAS     =('N_DIAS',     'sum'),
            PCT_ACTIVO =('PCT_ACTIVO', 'mean'),
        )
    cols = STATS_COLS + ['N_DIAS', 'PCT_ACTIVO']
    out = sub.set_index('CUARTO')[cols]
    return out.sort_index() if ordenar else out

print("Helpers visuales cargados.")

Helpers visuales cargados.


In [66]:
# ── Paleta de colores por mes ────────────────────────────────────────────────
_MES_COLORES = {
    1: '#4E79A7', 2: '#F28E2B', 3: '#E15759', 4: '#76B7B2',
    5: '#59A14F', 6: '#EDC948', 7: '#B07AA1', 8: '#FF9DA7',
    9: '#9C755F',10: '#BAB0AC',11: '#D4A6C8',12: '#86BCB6',
}

# Nota metodologica por tecnologia
_NOTA_TEC = {
    'Solar':   'Factor de recurso: fraccion de la capacidad instalada solar operando en cada cuarto.',
    'Eolica':  'Factor de recurso: fraccion de la capacidad instalada eolica operando en cada cuarto.',
    'Hidro':   'Factor de utilizacion: fraccion de la capacidad hidro despachada frecuentemente.',
    'Termica': 'Factor de utilizacion: fraccion de la capacidad termica despachada frecuentemente.',
}

# Umbrales de severidad operativa
UMBRAL_CRITICO  = 150   # MW / 15 min
UMBRAL_ATENCION =  50   # MW / 15 min

print(f"UMBRAL_CRITICO={UMBRAL_CRITICO} MW  |  UMBRAL_ATENCION={UMBRAL_ATENCION} MW")

UMBRAL_CRITICO=150 MW  |  UMBRAL_ATENCION=50 MW


# Sección 3 cont. — Widget de estación global

In [67]:
_MES_WIDGETS      = []
_UPDATE_FUNCTIONS = []

MESES_ACTIVOS = MESES_DISP[:]

def _on_estacion(change=None):
    global MESES_ACTIVOS
    v = w_estacion.value
    if v == 'verano':
        MESES_ACTIVOS = [m for m in MESES_DISP if m in MESES_VERANO]
    elif v == 'lluvioso':
        MESES_ACTIVOS = [m for m in MESES_DISP if m in MESES_LLUVIOSO]
    else:
        MESES_ACTIVOS = MESES_DISP[:]
    for w in _MES_WIDGETS:
        try:
            opts = [(MESES_NOMBRES[m], m) for m in MESES_ACTIVOS]
            prev = w.value
            w.options = opts
            w.value   = prev if prev in MESES_ACTIVOS else (MESES_ACTIVOS[0] if MESES_ACTIVOS else None)
        except Exception: pass
    for fn in _UPDATE_FUNCTIONS:
        try: fn(None)
        except Exception: pass

w_estacion = widgets.Dropdown(
    options=[('Todas','todas'),('Verano (Ene-May)','verano'),('Lluvioso (Jun-Dic)','lluvioso')],
    value='todas',
    description='Estacion:',
    style={'description_width':'80px'},
    layout=widgets.Layout(width='260px'),
)
w_estacion.observe(_on_estacion, names='value')
display(widgets.HBox([w_estacion]))
print("Estacion activa: todas  |  Cambia para filtrar meses en todos los bloques.")

Estacion activa: todas  |  Cambia para filtrar meses en todos los bloques.


# Sección 3 cont. — Funciones de plot compartidas

In [68]:
# ══════════════════════════════════════════════════════════════════════════════
# Sección 3 cont. — Funciones de plot compartidas
# _shared_perfil, _shared_envelope, _shared_rampas, _shared_maximos,
# _calcular_factores, _plot_perfil_meses
# ══════════════════════════════════════════════════════════════════════════════

_STAT_COLORS = {
    'MINIMA':    '#9C755F',
    'MODA_BAJA': '#4E79A7',
    'MEDIANA':   '#59A14F',
    'MODA_ALTA': '#E8A838',
    'MAXIMA':    '#E15759',
}


def _shared_perfil(factores_dict, cap, stats):
    """Perfil diario multi-stat. factores_dict = {stat: {mes: Series}}."""
    fig = go.Figure()
    for stat in stats:
        est = STAT_EST.get(stat, {})
        for mes in sorted(factores_dict.get(stat, {}).keys()):
            fa = factores_dict[stat][mes]
            fig.add_trace(go.Scatter(
                x=fa.index.tolist(), y=fa.tolist(), mode='lines',
                line=dict(color=_MES_COLORES.get(mes, '#fff'),
                          width=est.get('width', 2), dash=est.get('dash', 'solid')),
                opacity=est.get('opacity', 0.8),
                name=f'{MESES_NOMBRES[mes]} · {stat}',
                hovertemplate=(f'{MESES_NOMBRES[mes]} · {stat}'
                               f'<br>C-%{{x}}: <b>%{{y:.1%}}</b><extra></extra>'),
            ))
    _layout_oscuro(fig, y_title='Factor de generación')
    fig.update_yaxes(tickformat='.0%')
    fig.update_xaxes(range=[15, 2400], dtick=100)
    _agregar_amanecer_atardecer(fig)
    return fig


def _shared_envelope(factores_all, cap, color):
    """Envelope anual con 5 stats y banda MODA_BAJA-MODA_ALTA."""
    meses_orden = sorted({m for s in factores_all.values() for m in s})
    if not meses_orden:
        return go.Figure()
    nombres = [MESES_NOMBRES[m] for m in meses_orden]
    picos   = {s: [float(factores_all[s].get(m, pd.Series([0])).max()) for m in meses_orden]
               for s in STATS_COLS}
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=nombres + nombres[::-1],
        y=picos['MODA_ALTA'] + picos['MODA_BAJA'][::-1],
        fill='toself', fillcolor=hex_to_rgba(color, 0.12),
        line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False,
    ))
    for stat in STATS_COLS:
        est = STAT_EST[stat]
        fig.add_trace(go.Scatter(
            x=nombres, y=picos[stat], mode='lines+markers', name=stat,
            line=dict(color=color, width=est['width'], dash=est['dash']),
            marker=dict(size=6, opacity=est['opacity'], line=dict(color='#fff', width=0.8)),
            opacity=est['opacity'],
            hovertemplate='%{x}<br>' + stat + ': <b>%{y:.1%}</b><extra></extra>',
        ))
    for y_val, col, lbl in [
        (0.95, 'rgba(231,76,60,0.6)',  'Critico 95%'),
        (0.80, 'rgba(243,156,18,0.6)', 'Atencion 80%'),
    ]:
        fig.add_hline(y=y_val, line_color=col, line_dash='dot', line_width=1,
                      annotation_text=lbl,
                      annotation_font=dict(size=10,
                          color='#e74c3c' if '95' in lbl else '#f39c12'),
                      annotation_position='right')
    _layout_oscuro(fig, y_title='Factor de generacion (pico mensual)')
    fig.update_yaxes(tickformat='.0%', range=[0, 1.05])
    return fig


def _shared_rampas(ramp_df, filtros_solar, cap_actual, cap_objetivo,
                   meses, mes_detalle,
                   gen_solar_up_df=None, gen_solar_down_df=None):
    """
    Devuelve (fig_mensual, fig_diario).

    Cuando gen_solar_up_df / gen_solar_down_df están disponibles, la
    generacion solar en el cuarto de la rampa maxima aparece en el
    hover tooltip de las trazas proyectadas — sin ejes secundarios.
    """
    factor   = cap_objetivo / cap_actual if cap_actual > 0 else 1
    UC, UA   = UMBRAL_CRITICO, UMBRAL_ATENCION
    _has_gen = (gen_solar_up_df   is not None and not gen_solar_up_df.empty and
                gen_solar_down_df is not None and not gen_solar_down_df.empty)
    _up_cols   = set(gen_solar_up_df.columns)   if _has_gen else set()
    _down_cols = set(gen_solar_down_df.columns) if _has_gen else set()

    def _lookup_gen(df, valid_cols, filtros, m, dia):
        sub = df.copy()
        for c, v in {**filtros, 'MES': m, 'DIA': dia}.items():
            if c in valid_cols:
                sub = sub[sub[c] == v]
        return round(float(sub['GEN_SOLAR'].values[0]) * factor, 2) if not sub.empty else 0.0

    upA, dnA, upP, dnP, labels        = [], [], [], [], []
    gen_sol_up_mes, gen_sol_dn_mes     = [], []

    for m in meses:
        sub = ramp_df.copy()
        for col, val in {**filtros_solar, 'MES': m}.items():
            sub = sub[sub[col] == val]
        if sub.empty:
            continue
        _up_max = float(sub['RAMP_UP_MAX'].max())
        upA.append(_up_max)
        upP.append(round(_up_max * factor, 2))
        if _has_gen and _up_max > 0:
            _day_up = int(sub.loc[sub['RAMP_UP_MAX'].idxmax(), 'DIA'])
            gen_sol_up_mes.append(_lookup_gen(gen_solar_up_df, _up_cols, filtros_solar, m, _day_up))
        else:
            gen_sol_up_mes.append(0.0)

        _dn_min = float(sub['RAMP_DOWN_MAX'].min())
        dnA.append(_dn_min)
        dnP.append(round(_dn_min * factor, 2))
        if _has_gen and _dn_min < 0:
            _day_dn = int(sub.loc[sub['RAMP_DOWN_MAX'].idxmin(), 'DIA'])
            gen_sol_dn_mes.append(_lookup_gen(gen_solar_down_df, _down_cols, filtros_solar, m, _day_dn))
        else:
            gen_sol_dn_mes.append(0.0)
        labels.append(MESES_NOMBRES[m])

    def _evs(umbral):
        v = i = 0
        for m in meses:
            sub = ramp_df.copy()
            for col, val in {**filtros_solar, 'MES': m}.items():
                sub = sub[sub[col] == val]
            for _, row in sub.iterrows():
                up = abs(float(row['RAMP_UP_MAX'])) * factor
                dn = abs(float(row['RAMP_DOWN_MAX'])) * factor
                if up >= umbral or dn >= umbral:
                    if m in MESES_VERANO: v += 1
                    else:                 i += 1
        return v, i

    eA, eC = _evs(UA), _evs(UC)
    capS   = f'{cap_actual:.0f}'

    # ── Hover templates ───────────────────────────────────────────────────────
    _htmpl_up_actual  = '%{x} actual: <b>+%{y:.1f} MW</b><extra></extra>'
    _htmpl_dn_actual  = '%{x} actual: <b>%{y:.1f} MW</b><extra></extra>'
    if _has_gen:
        _htmpl_up_proy = '%{x} proy.: <b>+%{y:.1f} MW</b><br>☀ Solar: <b>%{customdata:.1f} MW</b><extra></extra>'
        _htmpl_dn_proy = '%{x} proy.: <b>%{y:.1f} MW</b><br>☀ Solar: <b>%{customdata:.1f} MW</b><extra></extra>'
    else:
        _htmpl_up_proy = '%{x} proy.: <b>+%{y:.1f} MW</b><extra></extra>'
        _htmpl_dn_proy = '%{x} proy.: <b>%{y:.1f} MW</b><extra></extra>'

    # ── fig_mes ───────────────────────────────────────────────────────────────
    fig_mes = go.Figure()

    if labels:
        fig_mes.add_trace(go.Scatter(
            x=labels, y=upA, mode='lines+markers',
            name=f'Ramp-up actual ({capS} MW)',
            line=dict(color='rgba(180,180,180,0.5)', width=1.5, dash='dot'),
            marker=dict(size=4),
            hovertemplate=_htmpl_up_actual))
        fig_mes.add_trace(go.Scatter(
            x=labels, y=dnA, mode='lines+markers',
            name=f'Ramp-down actual ({capS} MW)',
            line=dict(color='rgba(180,180,180,0.5)', width=1.5, dash='dot'),
            marker=dict(size=4),
            hovertemplate=_htmpl_dn_actual))
        fig_mes.add_trace(go.Scatter(
            x=labels, y=upP, mode='lines+markers',
            name=f'Ramp-up proy. ({cap_objetivo} MW)',
            line=dict(color='#2ecc71', width=2.5),
            marker=dict(size=7, color='#2ecc71', line=dict(color='#fff', width=1)),
            customdata=gen_sol_up_mes if _has_gen else None,
            hovertemplate=_htmpl_up_proy))
        fig_mes.add_trace(go.Scatter(
            x=labels, y=dnP, mode='lines+markers',
            name=f'Ramp-down proy. ({cap_objetivo} MW)',
            line=dict(color='#e74c3c', width=2.5),
            marker=dict(size=7, color='#e74c3c', line=dict(color='#fff', width=1)),
            customdata=gen_sol_dn_mes if _has_gen else None,
            hovertemplate=_htmpl_dn_proy))

    for y_val, col, lbl in [
        ( UA, 'rgba(243,156,18,0.5)', f'+{UA} MW · V:{eA[0]}/I:{eA[1]} dias'),
        (-UA, 'rgba(243,156,18,0.5)', f'-{UA} MW · V:{eA[0]}/I:{eA[1]} dias'),
        ( UC, 'rgba(231,76,60,0.7)',  f'+{UC} MW · V:{eC[0]}/I:{eC[1]} dias'),
        (-UC, 'rgba(231,76,60,0.7)',  f'-{UC} MW · V:{eC[0]}/I:{eC[1]} dias'),
    ]:
        fig_mes.add_hline(
            y=y_val, line_color=col,
            line_dash='dash' if abs(y_val) == UC else 'dot', line_width=1,
            annotation_text=lbl,
            annotation_font=dict(size=10,
                color='#e74c3c' if abs(y_val) == UC else '#f39c12'),
            annotation_position='right')
    _layout_oscuro(fig_mes, y_title='Rampa MW / 15 min')
    fig_mes.update_xaxes(title_text='Mes')
    fig_mes.update_layout(hovermode='x unified')

    # ── fig_dia ───────────────────────────────────────────────────────────────
    sub_d = ramp_df.copy()
    for col, val in {**filtros_solar, 'MES': mes_detalle}.items():
        sub_d = sub_d[sub_d[col] == val]
    dias  = sorted(sub_d['DIA'].unique()) if not sub_d.empty else []
    up_d  = [round(float(sub_d[sub_d['DIA']==d]['RAMP_UP_MAX'].max())   * factor, 2) for d in dias]
    dn_d  = [round(float(sub_d[sub_d['DIA']==d]['RAMP_DOWN_MAX'].min()) * factor, 2) for d in dias]
    cols_bar = [
        '#e74c3c' if max(abs(u), abs(dn)) >= UC
        else '#f39c12' if max(abs(u), abs(dn)) >= UA
        else '#2ecc71'
        for u, dn in zip(up_d, dn_d)
    ]

    gen_dia_up, gen_dia_dn = [], []
    if _has_gen and dias:
        _sg_up = gen_solar_up_df.copy()
        _sg_dn = gen_solar_down_df.copy()
        for c, v in {**filtros_solar, 'MES': mes_detalle}.items():
            if c in _up_cols:   _sg_up = _sg_up[_sg_up[c] == v]
            if c in _down_cols: _sg_dn = _sg_dn[_sg_dn[c] == v]
        for d in dias:
            r_up = _sg_up[_sg_up['DIA'] == d]
            gen_dia_up.append(round(float(r_up['GEN_SOLAR'].values[0]) * factor, 2) if not r_up.empty else 0.0)
            r_dn = _sg_dn[_sg_dn['DIA'] == d]
            gen_dia_dn.append(round(float(r_dn['GEN_SOLAR'].values[0]) * factor, 2) if not r_dn.empty else 0.0)

    if _has_gen and dias:
        _htmpl_bar_up = 'Dia %{x}: <b>+%{y:.1f} MW</b><br>☀ Solar: <b>%{customdata:.1f} MW</b><extra></extra>'
        _htmpl_bar_dn = 'Dia %{x}: <b>%{y:.1f} MW</b><br>☀ Solar: <b>%{customdata:.1f} MW</b><extra></extra>'
    else:
        _htmpl_bar_up = 'Dia %{x}: <b>+%{y:.1f} MW</b><extra></extra>'
        _htmpl_bar_dn = 'Dia %{x}: <b>%{y:.1f} MW</b><extra></extra>'

    fig_dia = go.Figure()
    if dias:
        fig_dia.add_trace(go.Bar(
            x=dias, y=up_d, name='Ramp-up',
            marker=dict(color=cols_bar, opacity=.90),
            customdata=gen_dia_up if _has_gen else None,
            hovertemplate=_htmpl_bar_up))
        fig_dia.add_trace(go.Bar(
            x=dias, y=dn_d, name='Ramp-down',
            marker=dict(color=cols_bar, opacity=.40),
            customdata=gen_dia_dn if _has_gen else None,
            hovertemplate=_htmpl_bar_dn))
    for y_val, col in [
        ( UA, 'rgba(243,156,18,0.6)'), (-UA, 'rgba(243,156,18,0.6)'),
        ( UC, 'rgba(231,76,60,0.75)'), (-UC, 'rgba(231,76,60,0.75)')
    ]:
        fig_dia.add_hline(y=y_val, line_color=col,
                          line_dash='dash' if abs(y_val) == UC else 'dot', line_width=1)
    _layout_oscuro(fig_dia, y_title='Rampa MW / 15 min')
    fig_dia.update_xaxes(title_text='Dia del mes', dtick=1)
    fig_dia.update_layout(barmode='overlay', bargap=.15)
    return fig_mes, fig_dia

def _shared_maximos(df_stats, filtros_solar, mes, cap_actual, cap_objetivo):
    """Maximos proyectados Solar."""
    factor = cap_objetivo / cap_actual if cap_actual > 0 else 1
    sub    = df_stats.copy()
    for col, val in {**filtros_solar, 'MES': mes}.items():
        sub = sub[sub[col] == val]
    st = _stats_por_cuarto(sub, {})
    if st.empty:
        return go.Figure()
    st    = st.loc[(st.index >= 15) & (st.index <= 2400)]
    xs    = st.index.tolist()
    y_act = st['MODA_ALTA'].tolist()
    y_pro = [v * factor for v in y_act]
    fig   = go.Figure()
    fig.add_trace(go.Scatter(
        x=xs + xs[::-1], y=y_pro + y_act[::-1], fill='toself',
        fillcolor='rgba(232,168,56,0.15)',
        line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', name='Incremento'))
    fig.add_trace(go.Scatter(
        x=xs, y=y_act, mode='lines',
        line=dict(color='rgba(180,180,180,0.7)', width=2, dash='dot'),
        name=f'Actual - {cap_actual:.0f} MW',
        hovertemplate='C-%{x}: %{y:.1f} MW<extra></extra>'))
    fig.add_trace(go.Scatter(
        x=xs, y=y_pro, mode='lines',
        line=dict(color='#E8A838', width=2.8),
        name=f'Proyectado - {cap_objetivo} MW',
        hovertemplate='C-%{x}: %{y:.1f} MW<extra></extra>'))
    _layout_oscuro(fig, y_title='MODA_ALTA Solar (MW)')
    fig.update_xaxes(range=[15, 2400])
    _agregar_amanecer_atardecer(fig)
    fig.update_layout(hovermode='x unified')
    return fig


def _calcular_factores(df, filtros_sin_mes, cap, stats=None):
    """Calcula factor de generacion por stat, mes y cuarto."""
    if stats is None:
        stats = ['MODA_ALTA']
    if cap <= 0:
        return {}
    resultado = {s: {} for s in stats}
    for mes in MESES_ACTIVOS:
        st = _stats_por_cuarto(df, {**filtros_sin_mes, 'MES': mes})
        if st.empty:
            continue
        st = st.loc[(st.index >= 15) & (st.index <= 2400)]
        for s in stats:
            if s in st.columns:
                f = (st[s] / cap).fillna(0.0)
                if not f.empty:
                    resultado[s][mes] = f
    return resultado



def _get_grupo_label(meses):
    """Auto-label para un grupo de meses."""
    vs = set(m for m in meses if m in MESES_VERANO)
    ll = set(m for m in meses if m in MESES_LLUVIOSO)
    if vs == MESES_VERANO and not ll:
        return 'Verano'
    if ll == MESES_LLUVIOSO and not vs:
        return 'Lluvioso'
    if len(meses) == len(MESES_DISP):
        return 'Todos'
    if len(meses) == 1:
        return MESES_NOMBRES[list(meses)[0]]
    return f'{len(meses)} meses'


def _plot_perfil_grupo(df, filtros_sin_mes, cap, meses, stats, modo='mes',
                       label='', dash_override=None):
    """
    Devuelve lista de traces para un grupo.
    modo='mes'      → 1 traza por (mes, stat).  Color=mes.
    modo='agregado' → 1 traza por stat (media).  Color=stat.
    dash_override   → fuerza el estilo de línea (útil para grupo 2).
    """
    traces = []
    if modo == 'mes':
        for mes in sorted(meses):
            st = _stats_por_cuarto(df, {**filtros_sin_mes, 'MES': mes})
            if st.empty: continue
            st = st.loc[(st.index >= 15) & (st.index <= 2400)]
            color_mes = _MES_COLORES.get(mes, '#fff')
            for stat in stats:
                if stat not in st.columns: continue
                est  = STAT_EST.get(stat, {})
                y    = (st[stat] / cap).fillna(0.0)
                dash = dash_override or est.get('dash', 'solid')
                traces.append(go.Scatter(
                    x=y.index.tolist(), y=y.tolist(), mode='lines',
                    line=dict(color=color_mes, width=est.get('width', 2.2), dash=dash),
                    name=f'{MESES_NOMBRES[mes]} · {stat}',
                    hovertemplate=(f'{MESES_NOMBRES[mes]} · {stat}'
                                   f'<br>C-%{{x}}: <b>%{{y:.1%}}</b><extra></extra>'),
                ))
    else:
        for stat in stats:
            est   = STAT_EST.get(stat, {})
            color = _STAT_COLORS.get(stat, '#fff')
            series_list = []
            for mes in sorted(meses):
                st = _stats_por_cuarto(df, {**filtros_sin_mes, 'MES': mes})
                if st.empty: continue
                st = st.loc[(st.index >= 15) & (st.index <= 2400)]
                if stat in st.columns:
                    series_list.append((st[stat] / cap).fillna(0.0))
            if not series_list: continue
            df_agg = pd.concat(series_list, axis=1).fillna(0.0)
            if stat == 'MINIMA':
                mean_s = df_agg.min(axis=1)
            elif stat == 'MAXIMA':
                mean_s = df_agg.max(axis=1)
            else:
                mean_s = df_agg.median(axis=1)  # mediana por cuarto sobre meses del periodo
            dash   = dash_override or est.get('dash', 'solid')
            n      = len(series_list)
            traces.append(go.Scatter(
                x=mean_s.index.tolist(), y=mean_s.tolist(), mode='lines',
                line=dict(color=color, width=est.get('width', 2.5), dash=dash),
                name=f'{label} · {stat} (n={n})',
                hovertemplate=(f'{label} · {stat}'
                               f'<br>C-%{{x}}: <b>%{{y:.1%}}</b><extra></extra>'),
            ))
    return traces


def _plot_perfil_meses(df, filtros_sin_mes, cap,
                       meses,  stats,  modo='mes',  label='',
                       meses2=None, stats2=None, modo2='mes', label2=''):
    """
    Ensambla 1 o 2 grupos en una figura.
    Grupo 2 usa dash='dash' para diferenciarse visualmente.
    """
    fig = go.Figure()
    lbl  = label  or _get_grupo_label(meses)
    for t in _plot_perfil_grupo(df, filtros_sin_mes, cap, meses, stats, modo, lbl):
        fig.add_trace(t)
    if meses2:
        lbl2 = label2 or _get_grupo_label(meses2)
        for t in _plot_perfil_grupo(df, filtros_sin_mes, cap, meses2,
                                    stats2 or stats, modo2, lbl2,
                                    dash_override='dash'):
            fig.add_trace(t)
    _layout_oscuro(fig, y_title='Factor de generación')
    fig.update_yaxes(tickformat='.0%')
    fig.update_xaxes(range=[15, 2400], dtick=100)
    _agregar_amanecer_atardecer(fig)
    return fig


print("Funciones compartidas OK: _shared_perfil, _shared_envelope, _shared_rampas, "
      "_shared_maximos, _calcular_factores, _plot_perfil_grupo, _plot_perfil_meses")

Funciones compartidas OK: _shared_perfil, _shared_envelope, _shared_rampas, _shared_maximos, _calcular_factores, _plot_perfil_grupo, _plot_perfil_meses


# Bloque 6 — Proyeccion por Tecnología

## 6a — Generación por Tecnología

In [69]:
# ══════════════════════════════════════════════════════════════════════════════
# Bloque 6a — Generación · Tecnología
# ══════════════════════════════════════════════════════════════════════════════
FF    = {}
out_6a = widgets.Output()

_MODO_OPTS = ['Por mes', 'Agregado']

def _actualizar_6a(change=None):
    global FF
    tec   = w_tec_6a.value
    cap   = _CAP_TEC.get(tec, 0)
    stats = list(w_stats_6a.value) or ['MODA_ALTA']
    m1    = list(w_meses1_6a.value) or MESES_ACTIVOS
    modo1 = 'agregado' if w_modo1_6a.value == 'Agregado' else 'mes'
    m2    = list(w_meses2_6a.value) if w_cmp_6a.value else None
    modo2 = 'agregado' if w_modo2_6a.value == 'Agregado' else 'mes'
    FF[tec] = {mes: (_stats_por_cuarto(df_tec, {'TECNOLOGIA': tec, 'MES': mes})
                     .loc[lambda s: (s.index>=15)&(s.index<=2400)]['MODA_ALTA']
                     .div(cap).fillna(0.0))
               for mes in MESES_DISP
               if not _stats_por_cuarto(df_tec, {'TECNOLOGIA': tec, 'MES': mes}).empty}
    with out_6a:
        out_6a.clear_output(wait=True)
        _plot_perfil_meses(df_tec, {'TECNOLOGIA': tec}, cap,
                           m1, stats, modo1,
                           meses2=m2, modo2=modo2).show()
        factores_all = _calcular_factores(df_tec, {'TECNOLOGIA': tec}, cap, STATS_COLS)
        _shared_envelope(factores_all, cap, tec_color.get(tec, '#fff')).show()

def _make_preset_buttons(w_meses, w_modo, callback):
    def _set(preset):
        meses_disp = [v for _, v in w_meses.options]
        if preset == 'todos':
            w_meses.value = tuple(meses_disp)
            w_modo.value  = 'Por mes'
        elif preset == 'verano':
            w_meses.value = tuple(m for m in meses_disp if m in MESES_VERANO)
            w_modo.value  = 'Agregado'
        elif preset == 'lluvioso':
            w_meses.value = tuple(m for m in meses_disp if m in MESES_LLUVIOSO)
            w_modo.value  = 'Agregado'
        callback()
    btn_t = widgets.Button(description='Todos',    button_style='', layout=widgets.Layout(width='80px', height='28px'))
    btn_v = widgets.Button(description='Verano',   button_style='', layout=widgets.Layout(width='80px', height='28px'))
    btn_l = widgets.Button(description='Lluvioso', button_style='', layout=widgets.Layout(width='80px', height='28px'))
    btn_t.on_click(lambda _: _set('todos'))
    btn_v.on_click(lambda _: _set('verano'))
    btn_l.on_click(lambda _: _set('lluvioso'))
    return widgets.HBox([btn_t, btn_v, btn_l])

# ── Widgets ───────────────────────────────────────────────────────────────────
w_tec_6a    = widgets.Dropdown(
    options=TECS_DISP, value='Solar',
    description='Tecnologia:', style={'description_width':'90px'},
    layout=widgets.Layout(width='240px'),
)
w_stats_6a  = widgets.SelectMultiple(
    options=STATS_COLS, value=['MODA_ALTA'],
    description='Stats:', style={'description_width':'50px'},
    layout=widgets.Layout(width='190px', height='110px'),
)
w_meses1_6a = widgets.SelectMultiple(
    options=[(MESES_NOMBRES[m], m) for m in MESES_DISP],
    value=tuple(MESES_DISP),
    layout=widgets.Layout(width='190px', height='110px'),
)
w_modo1_6a  = widgets.ToggleButtons(
    options=_MODO_OPTS, value='Por mes',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=widgets.Layout(width='195px'),
)
w_cmp_6a    = widgets.Checkbox(
    value=False, description='Comparar grupo 2',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='200px'),
)
w_meses2_6a = widgets.SelectMultiple(
    options=[(MESES_NOMBRES[m], m) for m in MESES_DISP],
    value=tuple(m for m in MESES_DISP if m in MESES_VERANO),
    layout=widgets.Layout(width='190px', height='110px'),
)
w_modo2_6a  = widgets.ToggleButtons(
    options=_MODO_OPTS, value='Agregado',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=widgets.Layout(width='195px'),
)
presets1_6a = _make_preset_buttons(w_meses1_6a, w_modo1_6a, _actualizar_6a)
presets2_6a = _make_preset_buttons(w_meses2_6a, w_modo2_6a, _actualizar_6a)

box_g2_6a = widgets.VBox([
    widgets.Label('Grupo 2:'),
    presets2_6a, w_meses2_6a, w_modo2_6a,
])
box_g2_6a.layout.display = 'none'

def _toggle_cmp_6a(change):
    box_g2_6a.layout.display = '' if w_cmp_6a.value else 'none'
    _actualizar_6a()

w_tec_6a.observe(_actualizar_6a,   names='value')
w_stats_6a.observe(_actualizar_6a, names='value')
w_meses1_6a.observe(_actualizar_6a, names='value')
w_modo1_6a.observe(_actualizar_6a,  names='value')
w_cmp_6a.observe(_toggle_cmp_6a,   names='value')
w_meses2_6a.observe(_actualizar_6a, names='value')
w_modo2_6a.observe(_actualizar_6a,  names='value')
_UPDATE_FUNCTIONS.append(_actualizar_6a)

display(widgets.VBox([
    widgets.HBox([w_tec_6a]),
    widgets.HBox([
        widgets.VBox([widgets.Label('Stats:'), w_stats_6a]),
        widgets.VBox([
            widgets.Label('Grupo 1:'),
            presets1_6a, w_meses1_6a, w_modo1_6a,
        ]),
    ]),
    w_cmp_6a,
    box_g2_6a,
    out_6a,
]))
_actualizar_6a()

## 6b — Rampas por Tecnología (Solar)

In [70]:
out_6b = widgets.Output()

def _actualizar_6b(change=None):
    tec        = w_tec_6b.value
    cap_actual = _CAP_TEC.get(tec, 0)
    cap_obj    = w_cap_6b.value if tec == 'Solar' else cap_actual   # Eolica no proyecta
    with out_6b:
        out_6b.clear_output(wait=True)
        fig_m, fig_d = _shared_rampas(
            ramp_df           = df_ramp_tec,
            filtros_solar     = {'TECNOLOGIA': tec},
            cap_actual        = cap_actual,
            cap_objetivo      = cap_obj,
            meses             = MESES_ACTIVOS,
            mes_detalle       = w_mes_6b.value,
            gen_solar_up_df   = df_gen_solar_up,
            gen_solar_down_df = df_gen_solar_down,
        )
        fig_m.show()
        fig_d.show()

def _on_tec_6b(change=None):
    w_cap_6b.layout.display = '' if w_tec_6b.value == 'Solar' else 'none'
    _actualizar_6b()

w_tec_6b = widgets.Dropdown(
    options=['Solar', 'Eolica'], value='Solar',
    description='Tecnologia:', style={'description_width':'90px'},
    layout=widgets.Layout(width='200px'),
)
w_mes_6b = widgets.Dropdown(
    options=[(MESES_NOMBRES[m], m) for m in MESES_ACTIVOS],
    value=MESES_ACTIVOS[0],
    description='Mes:', style={'description_width':'50px'},
    layout=widgets.Layout(width='190px'),
)
w_cap_6b = widgets.IntSlider(
    value=int(CAP_SOLAR_ACTUAL), min=int(CAP_SOLAR_ACTUAL), max=1600, step=10,
    description='Cap. Solar (MW):', style={'description_width':'130px'},
    layout=widgets.Layout(width='480px'),
)
_MES_WIDGETS.append(w_mes_6b)
w_tec_6b.observe(_on_tec_6b,     names='value')
w_mes_6b.observe(_actualizar_6b, names='value')
w_cap_6b.observe(_actualizar_6b, names='value')
_UPDATE_FUNCTIONS.append(_actualizar_6b)

display(widgets.VBox([widgets.HBox([w_tec_6b, w_mes_6b, w_cap_6b]), out_6b]))
_actualizar_6b()

## 6c — Maximos proyectados por Tecnología (Solar)

In [71]:
out_6c = widgets.Output()

def _actualizar_6c(change=None):
    with out_6c:
        out_6c.clear_output(wait=True)
        fig = _shared_maximos(
            df_stats      = df_tec,
            filtros_solar = {'TECNOLOGIA': 'Solar'},
            mes           = w_mes_6c.value,
            cap_actual    = CAP_SOLAR_ACTUAL,
            cap_objetivo  = w_cap_6c.value,
        )
        fig.show()

w_mes_6c = widgets.Dropdown(
    options=[(MESES_NOMBRES[m], m) for m in MESES_ACTIVOS],
    value=MESES_ACTIVOS[0],
    description='Mes:', style={'description_width':'50px'},
    layout=widgets.Layout(width='200px'),
)
w_cap_6c = widgets.IntSlider(
    value=int(CAP_SOLAR_ACTUAL), min=int(CAP_SOLAR_ACTUAL), max=1600, step=10,
    description='Cap. Solar (MW):', style={'description_width':'130px'},
    layout=widgets.Layout(width='500px'),
)
_MES_WIDGETS.append(w_mes_6c)
w_mes_6c.observe(_actualizar_6c, names='value')
w_cap_6c.observe(_actualizar_6c, names='value')
_UPDATE_FUNCTIONS.append(_actualizar_6c)

display(widgets.VBox([widgets.HBox([w_mes_6c, w_cap_6c]), out_6c]))
_actualizar_6c()

## 6d — Ventanas Horarias

Generación mediana por franja horaria en MW y % de capacidad instalada.

| Ventana | CUARTO |
|---|---|
| **MED AM** | 0800 – 1045 |
| **MAX** | 1100 – 1445 |
| **MED PM** | 1500 – 1645 |

In [72]:
# ══════════════════════════════════════════════════════════════════════════════
# Bloque 6d — Resumen PSSE: Ventanas Horarias por Planta
# Ventanas (orden cronológico):
#   MIN      23:00–06:00  (overnight baseline, cruza medianoche)
#   MED_AM   08:00–10:45
#   MAX      11:00–14:45
#   MED_PM   15:00–16:45
#   MED_NOCT 19:00–22:00
# Columnas: todas las ventanas (MW + %) | Cap.(MW) | Tecnología | Zona | Nodo
# ══════════════════════════════════════════════════════════════════════════════

VENTANAS_DEF_6D = {
    'MIN':      None,           # especial: CUARTO >= 2300 OR <= 600
    'MED_AM':   (800,  1045),
    'MAX':      (1100, 1445),
    'MED_PM':   (1500, 1645),
    'MED_NOCT': (1900, 2200),
}


def _calc_ventanas_planta(meses_sel):
    """Resumen PSSE — MW y % de cap. instalada por ventana horaria, nivel Planta.
    La misma lógica que _calc_ventanas original: agrupa por PLANTA+MES+CUARTO
    (suma de MEDIANA), luego promedia entre meses y toma el máximo dentro de
    cada ventana.  El nombre 'MIN' es solo un rótulo; el cálculo es idéntico.
    """
    df_f = df_planta[df_planta['MES'].isin(meses_sel)].copy()
    df_agg = (
        df_f
        .groupby(['PLANTA', 'MES', 'CUARTO'])['MEDIANA'].sum()
        .reset_index()
        .groupby(['PLANTA', 'CUARTO'])['MEDIANA'].mean()
        .reset_index()
    )
    rows = []
    for planta in sorted(df_agg['PLANTA'].dropna().unique()):
        cap = _CAP_PLANTA.get(planta, 0)
        if cap <= 0:
            continue
        grp = df_agg[df_agg['PLANTA'] == planta]
        row = {'Planta': planta, '_cap': cap}
        for vname, vrange in VENTANAS_DEF_6D.items():
            if vrange is None:                              # MIN: cruza medianoche
                mask = (grp['CUARTO'] >= 2300) | (grp['CUARTO'] <= 600)
            else:
                v0, v1 = vrange
                mask = (grp['CUARTO'] >= v0) & (grp['CUARTO'] <= v1)
            val = grp.loc[mask, 'MEDIANA'].max() if mask.any() else 0.0
            row[f'{vname}_MW'] = round(val, 1)
            row[f'{vname}_%']  = round(val / cap * 100, 1) if cap > 0 else 0.0
        rows.append(row)

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows).set_index('Planta')

    # Fila TOTAL
    cap_tot = result['_cap'].sum()
    total   = {'_cap': cap_tot}
    for vname in VENTANAS_DEF_6D:
        mw = result[f'{vname}_MW'].sum()
        total[f'{vname}_MW'] = round(mw, 1)
        total[f'{vname}_%']  = round(mw / cap_tot * 100, 1) if cap_tot > 0 else 0.0
    result.loc['── TOTAL ──'] = total

    # Columnas de metadatos (vacías para la fila TOTAL)
    result['Tecnología'] = result.index.map(lambda p: _TEC_PLANTA.get(p,  ''))
    result['Zona']       = result.index.map(lambda p: _ZONA_PLANTA.get(p, ''))
    result['Nodo']       = result.index.map(lambda p: _NODO_PLANTA.get(p,  ''))

    calc_cols = [f'{v}_{s}' for v in VENTANAS_DEF_6D for s in ('MW', '%')]
    return result[calc_cols + ['_cap', 'Tecnología', 'Zona', 'Nodo']].rename(
        columns={'_cap': 'Cap.(MW)'}
    )


# ── Columnas para estilos ──────────────────────────────────────────────────────
_NUM_COLS_6D  = (
    [f'{v}_{s}' for v in VENTANAS_DEF_6D for s in ('MW', '%')] + ['Cap.(MW)']
)
_META_COLS_6D = ['Tecnología', 'Zona', 'Nodo']

# ── Widgets ───────────────────────────────────────────────────────────────────
w_est_6d = widgets.ToggleButtons(
    options=['Todas', 'Verano (Dic–Abr)', 'Lluvioso (May–Nov)'],
    value='Todas',
    style={'button_width': '155px'},
)
out_6d = widgets.Output()


def _actualizar_6d(change=None):
    est = w_est_6d.value
    if est == 'Verano (Dic–Abr)':
        meses, label = list(MESES_VERANO),    'Verano (Dic–Abr)'
    elif est == 'Lluvioso (May–Nov)':
        meses, label = list(MESES_LLUVIOSO),  'Lluvioso (May–Nov)'
    else:
        meses, label = MESES_DISP,             'Todas las estaciones'

    df_result = _calc_ventanas_planta(meses)

    with out_6d:
        out_6d.clear_output(wait=True)
        if df_result.empty:
            print('Sin datos para la selección.')
            return

        num_cols  = [c for c in _NUM_COLS_6D  if c in df_result.columns]
        meta_cols = [c for c in _META_COLS_6D if c in df_result.columns]

        def _fmt(x):
            if isinstance(x, float) and not pd.isna(x):
                return f'{x:.1f}'
            if pd.isna(x):
                return ''
            return str(x)

        print(f'── Resumen PSSE · {label} ──')
        display(
            df_result.style
            .format(_fmt)
            .set_properties(
                subset=num_cols,
                **{'text-align': 'right', 'font-size': '13px', 'padding': '5px 14px'}
            )
            .set_properties(
                subset=meta_cols,
                **{'text-align': 'left', 'font-size': '13px', 'padding': '5px 14px'}
            )
            .set_table_styles([
                {'selector': 'th',
                 'props': [('background-color', '#2C3E50'), ('color', 'white'),
                           ('text-align', 'center'), ('padding', '7px 14px')]},
                {'selector': 'tr:last-child td',
                 'props': [('font-weight', 'bold'),
                           ('background-color', '#EBF5FB'),
                           ('border-top', '2px solid #2C3E50')]},
            ])
        )


w_est_6d.observe(_actualizar_6d, names='value')

display(widgets.VBox([
    widgets.HBox([widgets.HTML('<b>Estación:&nbsp;&nbsp;</b>'), w_est_6d]),
    out_6d,
]))
_actualizar_6d()


# Bloque 7 — Analisis por Nodo

## 7a — Generación por Nodo

In [73]:
# ══════════════════════════════════════════════════════════════════════════════
# Bloque 7a — Generación · Nodo
# ══════════════════════════════════════════════════════════════════════════════
out_7a = widgets.Output()

def _actualizar_7a(change=None):
    nodo  = w_nodo_7a.value
    tec   = w_tec_7a.value
    cap   = _CAP_NODO_TEC.get((nodo, tec), 0)
    stats = list(w_stats_7a.value) or ['MODA_ALTA']
    m1    = list(w_meses1_7a.value) or MESES_ACTIVOS
    modo1 = 'agregado' if w_modo1_7a.value == 'Agregado' else 'mes'
    m2    = list(w_meses2_7a.value) if w_cmp_7a.value else None
    modo2 = 'agregado' if w_modo2_7a.value == 'Agregado' else 'mes'
    if cap <= 0:
        with out_7a:
            out_7a.clear_output(wait=True)
            print(f'Sin capacidad para {nodo} / {tec}.')
        return
    with out_7a:
        out_7a.clear_output(wait=True)
        filtros = {'NODO': nodo, 'TECNOLOGIA': tec}
        _plot_perfil_meses(df_nodo, filtros, cap,
                           m1, stats, modo1,
                           meses2=m2, modo2=modo2).show()
        factores_all = _calcular_factores(df_nodo, filtros, cap, STATS_COLS)
        _shared_envelope(factores_all, cap, tec_color.get(tec, '#fff')).show()

def _on_nodo_change(change=None):
    nodo = w_nodo_7a.value
    tecs = NODO_TECS_DISP.get(nodo, TECS_DISP)
    w_tec_7a.options = tecs
    if w_tec_7a.value not in tecs:
        w_tec_7a.value = tecs[0] if tecs else None
    _actualizar_7a()

presets1_7a_ref = [None]
presets2_7a_ref = [None]

w_nodo_7a   = widgets.Dropdown(
    options=NODOS_DISP, value=NODOS_DISP[0],
    description='Nodo:', style={'description_width':'60px'},
    layout=widgets.Layout(width='180px'),
)
w_tec_7a    = widgets.Dropdown(
    options=NODO_TECS_DISP.get(NODOS_DISP[0], TECS_DISP),
    value=NODO_TECS_DISP.get(NODOS_DISP[0], TECS_DISP)[0],
    description='Tec:', style={'description_width':'40px'},
    layout=widgets.Layout(width='160px'),
)
w_stats_7a  = widgets.SelectMultiple(
    options=STATS_COLS, value=['MODA_ALTA'],
    description='Stats:', style={'description_width':'50px'},
    layout=widgets.Layout(width='190px', height='110px'),
)
w_meses1_7a = widgets.SelectMultiple(
    options=[(MESES_NOMBRES[m], m) for m in MESES_DISP],
    value=tuple(MESES_DISP),
    layout=widgets.Layout(width='190px', height='110px'),
)
w_modo1_7a  = widgets.ToggleButtons(
    options=_MODO_OPTS, value='Por mes',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=widgets.Layout(width='195px'),
)
w_cmp_7a    = widgets.Checkbox(
    value=False, description='Comparar grupo 2',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='200px'),
)
w_meses2_7a = widgets.SelectMultiple(
    options=[(MESES_NOMBRES[m], m) for m in MESES_DISP],
    value=tuple(m for m in MESES_DISP if m in MESES_VERANO),
    layout=widgets.Layout(width='190px', height='110px'),
)
w_modo2_7a  = widgets.ToggleButtons(
    options=_MODO_OPTS, value='Agregado',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=widgets.Layout(width='195px'),
)
presets1_7a = _make_preset_buttons(w_meses1_7a, w_modo1_7a, _actualizar_7a)
presets2_7a = _make_preset_buttons(w_meses2_7a, w_modo2_7a, _actualizar_7a)

box_g2_7a = widgets.VBox([
    widgets.Label('Grupo 2:'),
    presets2_7a, w_meses2_7a, w_modo2_7a,
])
box_g2_7a.layout.display = 'none'

def _toggle_cmp_7a(change):
    box_g2_7a.layout.display = '' if w_cmp_7a.value else 'none'
    _actualizar_7a()

w_nodo_7a.observe(_on_nodo_change,   names='value')
w_tec_7a.observe(_actualizar_7a,     names='value')
w_stats_7a.observe(_actualizar_7a,   names='value')
w_meses1_7a.observe(_actualizar_7a,  names='value')
w_modo1_7a.observe(_actualizar_7a,   names='value')
w_cmp_7a.observe(_toggle_cmp_7a,     names='value')
w_meses2_7a.observe(_actualizar_7a,  names='value')
w_modo2_7a.observe(_actualizar_7a,   names='value')
_UPDATE_FUNCTIONS.append(_actualizar_7a)

display(widgets.VBox([
    widgets.HBox([w_nodo_7a, w_tec_7a]),
    widgets.HBox([
        widgets.VBox([widgets.Label('Stats:'), w_stats_7a]),
        widgets.VBox([
            widgets.Label('Grupo 1:'),
            presets1_7a, w_meses1_7a, w_modo1_7a,
        ]),
    ]),
    w_cmp_7a,
    box_g2_7a,
    out_7a,
]))
_actualizar_7a()

## 7b — Rampas por Nodo (Solar)

In [74]:
out_7b = widgets.Output()

def _actualizar_7b(change=None):
    nodo       = w_nodo_7b.value
    tec        = w_tec_7b.value
    cap_actual = _CAP_NODO_TEC.get((nodo, tec), 0)
    if cap_actual <= 0:
        with out_7b:
            out_7b.clear_output(wait=True)
            print(f'Sin capacidad {tec} para nodo {nodo}.')
        return
    cap_obj = w_cap_7b.value if tec == 'Solar' else cap_actual   # Eolica no proyecta
    with out_7b:
        out_7b.clear_output(wait=True)
        fig_m, fig_d = _shared_rampas(
            ramp_df           = df_ramp_nodo,
            filtros_solar     = {'NODO': nodo, 'TECNOLOGIA': tec},
            cap_actual        = cap_actual,
            cap_objetivo      = cap_obj,
            meses             = MESES_ACTIVOS,
            mes_detalle       = w_mes_7b.value,
            gen_solar_up_df   = df_gen_solar_up,
            gen_solar_down_df = df_gen_solar_down,
        )
        fig_m.show()
        fig_d.show()

def _on_nodo_7b(change=None):
    nodo = w_nodo_7b.value
    tecs = [t for t in ['Solar', 'Eolica'] if _CAP_NODO_TEC.get((nodo, t), 0) > 0]
    w_tec_7b.options = tecs or ['Solar']
    if w_tec_7b.value not in (tecs or ['Solar']):
        w_tec_7b.value = (tecs or ['Solar'])[0]
    _on_tec_7b()

def _on_tec_7b(change=None):
    nodo = w_nodo_7b.value
    tec  = w_tec_7b.value
    cap_actual = _CAP_NODO_TEC.get((nodo, tec), 0)
    if tec == 'Solar' and cap_actual > 0:
        w_cap_7b.min   = int(cap_actual)
        w_cap_7b.max   = max(1600, int(cap_actual * 3))
        w_cap_7b.value = int(cap_actual)
        w_cap_7b.layout.display = ''
    else:
        w_cap_7b.layout.display = 'none'
    _actualizar_7b()

# Nodos con Solar o Eolica
_nodos_re_7b = [n for n in NODOS_DISP
                if _CAP_NODO_TEC.get((n,'Solar'),0)>0 or _CAP_NODO_TEC.get((n,'Eolica'),0)>0]
_nodo0_7b    = _nodos_re_7b[0] if _nodos_re_7b else NODOS_DISP[0]
_tecs0_7b    = [t for t in ['Solar','Eolica'] if _CAP_NODO_TEC.get((_nodo0_7b,t),0)>0] or ['Solar']
_cap0_7b     = _CAP_NODO_TEC.get((_nodo0_7b, _tecs0_7b[0]), CAP_SOLAR_ACTUAL)

w_nodo_7b = widgets.Dropdown(
    options=_nodos_re_7b or NODOS_DISP, value=_nodo0_7b,
    description='Nodo:', style={'description_width':'60px'},
    layout=widgets.Layout(width='180px'),
)
w_tec_7b  = widgets.Dropdown(
    options=_tecs0_7b, value=_tecs0_7b[0],
    description='Tec:', style={'description_width':'40px'},
    layout=widgets.Layout(width='140px'),
)
w_mes_7b  = widgets.Dropdown(
    options=[(MESES_NOMBRES[m], m) for m in MESES_ACTIVOS],
    value=MESES_ACTIVOS[0],
    description='Mes:', style={'description_width':'50px'},
    layout=widgets.Layout(width='190px'),
)
w_cap_7b  = widgets.IntSlider(
    value=int(_cap0_7b), min=int(_cap0_7b),
    max=max(1600, int(_cap0_7b*3)), step=10,
    description='Cap. Solar (MW):', style={'description_width':'130px'},
    layout=widgets.Layout(width='460px'),
)
if _tecs0_7b[0] != 'Solar':
    w_cap_7b.layout.display = 'none'

_MES_WIDGETS.append(w_mes_7b)
w_nodo_7b.observe(_on_nodo_7b,   names='value')
w_tec_7b.observe(_on_tec_7b,     names='value')
w_mes_7b.observe(_actualizar_7b, names='value')
w_cap_7b.observe(_actualizar_7b, names='value')
_UPDATE_FUNCTIONS.append(_actualizar_7b)

display(widgets.VBox([widgets.HBox([w_nodo_7b, w_tec_7b, w_mes_7b, w_cap_7b]), out_7b]))
_actualizar_7b()

## 7c — Maximos proyectados por Nodo (Solar)

In [75]:
out_7c = widgets.Output()

def _actualizar_7c(change=None):
    nodo      = w_nodo_7c.value
    cap_solar = _CAP_NODO_TEC.get((nodo, 'Solar'), 0)
    if cap_solar <= 0:
        with out_7c:
            out_7c.clear_output(wait=True)
            print(f'Sin capacidad Solar para nodo {nodo}.')
        return
    with out_7c:
        out_7c.clear_output(wait=True)
        fig = _shared_maximos(
            df_stats      = df_nodo,
            filtros_solar = {'NODO': nodo, 'TECNOLOGIA': 'Solar'},
            mes           = w_mes_7c.value,
            cap_actual    = cap_solar,
            cap_objetivo  = w_cap_7c.value,
        )
        fig.show()

def _on_nodo_7c(change=None):
    nodo      = w_nodo_7c.value
    cap_solar = _CAP_NODO_TEC.get((nodo, 'Solar'), 0)
    if cap_solar > 0:
        w_cap_7c.min   = int(cap_solar)
        w_cap_7c.max   = max(1600, int(cap_solar * 3))
        w_cap_7c.value = int(cap_solar)
    _actualizar_7c()

_nodos_solar_7c = [n for n in NODOS_DISP if _CAP_NODO_TEC.get((n,'Solar'),0)>0]
_nodo0_7c       = _nodos_solar_7c[0] if _nodos_solar_7c else NODOS_DISP[0]
_cap0_7c        = _CAP_NODO_TEC.get((_nodo0_7c,'Solar'), CAP_SOLAR_ACTUAL)

w_nodo_7c = widgets.Dropdown(
    options=_nodos_solar_7c or NODOS_DISP,
    value=_nodo0_7c,
    description='Nodo:', style={'description_width':'60px'},
    layout=widgets.Layout(width='180px'),
)
w_mes_7c  = widgets.Dropdown(
    options=[(MESES_NOMBRES[m], m) for m in MESES_ACTIVOS],
    value=MESES_ACTIVOS[0],
    description='Mes:', style={'description_width':'50px'},
    layout=widgets.Layout(width='200px'),
)
w_cap_7c  = widgets.IntSlider(
    value=int(_cap0_7c), min=int(_cap0_7c),
    max=max(1600, int(_cap0_7c*3)), step=10,
    description='Cap. Solar (MW):', style={'description_width':'130px'},
    layout=widgets.Layout(width='500px'),
)
_MES_WIDGETS.append(w_mes_7c)
w_nodo_7c.observe(_on_nodo_7c,   names='value')
w_mes_7c.observe(_actualizar_7c, names='value')
w_cap_7c.observe(_actualizar_7c, names='value')
_UPDATE_FUNCTIONS.append(_actualizar_7c)

display(widgets.VBox([widgets.HBox([w_nodo_7c, w_mes_7c, w_cap_7c]), out_7c]))
_actualizar_7c()

# Bloque 8 — Analisis por Planta

## 8a — Generación por Planta

In [76]:
# ══════════════════════════════════════════════════════════════════════════════
# Bloque 8a — Generación · Planta
# ══════════════════════════════════════════════════════════════════════════════
out_8a = widgets.Output()

def _info_planta_html(planta):
    tec  = _TEC_PLANTA.get(planta, '') or ''
    col  = tec_color.get(tec, '#fff')
    nodo = _NODO_PLANTA.get(planta, '') or '?'
    zona = _ZONA_PLANTA.get(planta, '') or '?'
    cap  = float(_CAP_PLANTA.get(planta, 0) or 0)
    return (f"<span style='font-size:12px;color:#888'>"
            f"<b style='color:{col}'>{tec}</b> &nbsp;&middot;&nbsp; "
            f"Nodo: <b>{nodo}</b> &nbsp;&middot;&nbsp; "
            f"Zona: <b>{zona}</b> &nbsp;&middot;&nbsp; "
            f"Cap. instalada: <b>{cap:.2f} MW</b></span>")

def _actualizar_8a(change=None):
    planta = w_planta_8a.value
    tec    = _TEC_PLANTA.get(planta, '') or ''
    cap    = float(_CAP_PLANTA.get(planta, 0) or 0)
    lbl_info_8a.value = _info_planta_html(planta)
    stats = list(w_stats_8a.value) or ['MODA_ALTA']
    m1    = list(w_meses1_8a.value) or MESES_ACTIVOS
    modo1 = 'agregado' if w_modo1_8a.value == 'Agregado' else 'mes'
    m2    = list(w_meses2_8a.value) if w_cmp_8a.value else None
    modo2 = 'agregado' if w_modo2_8a.value == 'Agregado' else 'mes'
    if cap <= 0:
        with out_8a:
            out_8a.clear_output(wait=True)
            print(f'Sin capacidad para {planta}.')
        return
    with out_8a:
        out_8a.clear_output(wait=True)
        filtros = {'PLANTA': planta}
        _plot_perfil_meses(df_planta, filtros, cap,
                           m1, stats, modo1,
                           meses2=m2, modo2=modo2).show()
        factores_all = _calcular_factores(df_planta, filtros, cap, STATS_COLS)
        _shared_envelope(factores_all, cap, tec_color.get(tec, '#fff')).show()

w_planta_8a = widgets.Dropdown(
    options=PLANTAS_DISP, value=PLANTAS_DISP[0],
    description='Planta:', style={'description_width':'60px'},
    layout=widgets.Layout(width='320px'),
)
lbl_info_8a = widgets.HTML(value='')
w_stats_8a  = widgets.SelectMultiple(
    options=STATS_COLS, value=['MODA_ALTA'],
    description='Stats:', style={'description_width':'50px'},
    layout=widgets.Layout(width='190px', height='110px'),
)
w_meses1_8a = widgets.SelectMultiple(
    options=[(MESES_NOMBRES[m], m) for m in MESES_DISP],
    value=tuple(MESES_DISP),
    layout=widgets.Layout(width='190px', height='110px'),
)
w_modo1_8a  = widgets.ToggleButtons(
    options=_MODO_OPTS, value='Por mes',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=widgets.Layout(width='195px'),
)
w_cmp_8a    = widgets.Checkbox(
    value=False, description='Comparar grupo 2',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='200px'),
)
w_meses2_8a = widgets.SelectMultiple(
    options=[(MESES_NOMBRES[m], m) for m in MESES_DISP],
    value=tuple(m for m in MESES_DISP if m in MESES_VERANO),
    layout=widgets.Layout(width='190px', height='110px'),
)
w_modo2_8a  = widgets.ToggleButtons(
    options=_MODO_OPTS, value='Agregado',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=widgets.Layout(width='195px'),
)
presets1_8a = _make_preset_buttons(w_meses1_8a, w_modo1_8a, _actualizar_8a)
presets2_8a = _make_preset_buttons(w_meses2_8a, w_modo2_8a, _actualizar_8a)

box_g2_8a = widgets.VBox([
    widgets.Label('Grupo 2:'),
    presets2_8a, w_meses2_8a, w_modo2_8a,
])
box_g2_8a.layout.display = 'none'

def _toggle_cmp_8a(change):
    box_g2_8a.layout.display = '' if w_cmp_8a.value else 'none'
    _actualizar_8a()

w_planta_8a.observe(_actualizar_8a, names='value')
w_stats_8a.observe(_actualizar_8a,  names='value')
w_meses1_8a.observe(_actualizar_8a, names='value')
w_modo1_8a.observe(_actualizar_8a,  names='value')
w_cmp_8a.observe(_toggle_cmp_8a,    names='value')
w_meses2_8a.observe(_actualizar_8a, names='value')
w_modo2_8a.observe(_actualizar_8a,  names='value')
_UPDATE_FUNCTIONS.append(_actualizar_8a)

display(widgets.VBox([
    widgets.HBox([w_planta_8a]),
    lbl_info_8a,
    widgets.HBox([
        widgets.VBox([widgets.Label('Stats:'), w_stats_8a]),
        widgets.VBox([
            widgets.Label('Grupo 1:'),
            presets1_8a, w_meses1_8a, w_modo1_8a,
        ]),
    ]),
    w_cmp_8a,
    box_g2_8a,
    out_8a,
]))
_actualizar_8a()

# Bloque 9 — Respuesta historica vs requerida · Hidro + Termica

In [77]:
out_resp = widgets.Output()

def _calcular_respuesta_mensual(driver, cap_obj):
    cap_actual = _CAP_TEC.get(driver, 0)
    factor = cap_obj / cap_actual if cap_actual > 0 else 1
    meses  = sorted(df_ramp_tec['MES'].unique())
    rows   = []
    for mes in meses:
        drv_mes = df_ramp_tec[
            (df_ramp_tec['TECNOLOGIA'] == driver) &
            (df_ramp_tec['MES']        == mes)
        ].set_index('DIA')
        if drv_mes.empty: continue
        dia_peor   = drv_mes['RAMP_DOWN_MAX'].abs().idxmax()
        drv_hist   = float(drv_mes.loc[dia_peor, 'RAMP_DOWN_MAX'])
        drv_proy   = abs(drv_hist) * factor
        hidro_mes  = df_ramp_tec[(df_ramp_tec['TECNOLOGIA']=='Hidro')&(df_ramp_tec['MES']==mes)].set_index('DIA')
        hidro_hist = float(hidro_mes.loc[dia_peor, 'RAMP_UP_MAX']) if dia_peor in hidro_mes.index else 0.0
        term_mes   = df_ramp_tec[(df_ramp_tec['TECNOLOGIA']=='Termica')&(df_ramp_tec['MES']==mes)].set_index('DIA')
        term_hist  = float(term_mes.loc[dia_peor, 'RAMP_UP_MAX']) if dia_peor in term_mes.index else 0.0
        hidro_req  = hidro_hist * factor
        term_req   = term_hist  * factor
        resp_total = hidro_req  + term_req
        brecha     = drv_proy - resp_total
        rows.append({'MES':mes,'NOMBRE':MESES_NOMBRES[mes],'DIA_PEOR':dia_peor,
                     'DRV_PROY':round(drv_proy,2),'HIDRO_HIST':round(hidro_hist,2),
                     'TERMICA_HIST':round(term_hist,2),'HIDRO_REQ':round(hidro_req,2),
                     'TERMICA_REQ':round(term_req,2),'BRECHA':round(brecha,2)})
    return pd.DataFrame(rows)

def _plot_respuesta_mensual(driver, cap_obj):
    cap_actual = _CAP_TEC.get(driver, 0)
    df_r       = _calcular_respuesta_mensual(driver, cap_obj)
    if df_r.empty: print('Sin datos.'); return
    nombres   = df_r['NOMBRE'].tolist()
    drv_color = tec_color.get(driver, '#E8A838')
    fig = go.Figure()
    fig.add_trace(go.Bar(name=f'{driver} exige ({cap_obj} MW)', x=nombres,
        y=df_r['DRV_PROY'].tolist(), marker_color=hex_to_rgba(drv_color, 0.90),
        hovertemplate='%{x}<br>'+driver+' exige: <b>%{y:.1f} MW</b><extra></extra>'))
    fig.add_trace(go.Bar(name=f'Hidro real ({cap_actual:.0f} MW)', x=nombres,
        y=df_r['HIDRO_HIST'].tolist(),
        marker_color='rgba(55,138,221,0.50)',
        marker_line=dict(color='rgba(55,138,221,0.9)',width=1.5),
        hovertemplate='%{x}<br>Hidro real: <b>%{y:.1f} MW</b><extra></extra>'))
    fig.add_trace(go.Bar(name=f'Hidro requerida ({cap_obj} MW)', x=nombres,
        y=df_r['HIDRO_REQ'].tolist(), marker_color='rgba(55,138,221,0.90)',
        hovertemplate='%{x}<br>Hidro req: <b>%{y:.1f} MW</b><extra></extra>'))
    fig.add_trace(go.Bar(name=f'Termica real ({cap_actual:.0f} MW)', x=nombres,
        y=df_r['TERMICA_HIST'].tolist(),
        marker_color='rgba(216,90,48,0.50)',
        marker_line=dict(color='rgba(216,90,48,0.9)',width=1.5),
        hovertemplate='%{x}<br>Termica real: <b>%{y:.1f} MW</b><extra></extra>'))
    fig.add_trace(go.Bar(name=f'Termica requerida ({cap_obj} MW)', x=nombres,
        y=df_r['TERMICA_REQ'].tolist(), marker_color='rgba(216,90,48,0.90)',
        hovertemplate='%{x}<br>Termica req: <b>%{y:.1f} MW</b><extra></extra>'))
    brechas = df_r['BRECHA'].tolist()
    col_br = ['#e74c3c' if b>UMBRAL_CRITICO else '#f39c12' if b>0 else '#2ecc71' for b in brechas]
    fig.add_trace(go.Scatter(name=f'Brecha ({driver} - Respuesta)', x=nombres, y=brechas,
        mode='markers+text',
        marker=dict(symbol='diamond',size=10,color=col_br,line=dict(color='#fff',width=1)),
        text=[f'{b:+.0f}' for b in brechas], textposition='top center', textfont=dict(size=9),
        hovertemplate='%{x}<br>Brecha: <b>%{y:+.1f} MW</b><extra></extra>'))
    for y_val, col, dash, texto in [
        (UMBRAL_ATENCION,'rgba(243,156,18,0.5)','dot', f'Atencion {UMBRAL_ATENCION} MW'),
        (UMBRAL_CRITICO, 'rgba(231,76,60,0.6)', 'dash',f'Critico {UMBRAL_CRITICO} MW'),
    ]:
        fig.add_hline(y=y_val, line_color=col, line_dash=dash, line_width=1,
                      annotation_text=texto,
                      annotation_font=dict(size=10,
                          color='#e74c3c' if 'Critico' in texto else '#f39c12'),
                      annotation_position='right')
    _layout_oscuro(fig, y_title='MW / 15 min (peor dia del mes)', height=520)
    fig.update_layout(barmode='group', bargap=0.15)
    fig.show()
    items = []
    for _, row in df_r.iterrows():
        sev = (SEV_CRITICO if row['BRECHA']>UMBRAL_CRITICO else
               SEV_ATENCION if row['BRECHA']>0 else SEV_OK)
        items.append((sev,
            f'{row["NOMBRE"]} (dia {int(row["DIA_PEOR"])})  |  '
            f'{driver} exige <b>{row["DRV_PROY"]:.1f} MW</b>  |  '
            f'Hidro real <b>{row["HIDRO_HIST"]:.1f}</b> -> req <b>{row["HIDRO_REQ"]:.1f} MW</b>  |  '
            f'Termica real <b>{row["TERMICA_HIST"]:.1f}</b> -> req <b>{row["TERMICA_REQ"]:.1f} MW</b>  |  '
            f'brecha <b>{row["BRECHA"]:+.1f} MW</b>'))
    _resumen_critico(items, titulo=f'Respuesta real vs requerida · {driver} @ {cap_obj} MW')

w_tec_resp = widgets.Dropdown(
    options=['Solar', 'Eolica'], value='Solar',
    description='Tecnologia:', style={'description_width':'90px'},
    layout=widgets.Layout(width='200px'),
)
w_cap_resp = widgets.IntSlider(
    value=int(CAP_SOLAR_ACTUAL), min=int(CAP_SOLAR_ACTUAL), max=1600, step=50,
    description='Cap. proyectada (MW):', style={'description_width':'180px'},
    layout=widgets.Layout(width='540px'),
)

def _actualizar_resp(change=None):
    driver     = w_tec_resp.value
    cap_actual = _CAP_TEC.get(driver, 0)
    cap_obj    = w_cap_resp.value if driver == 'Solar' else cap_actual
    with out_resp:
        out_resp.clear_output(wait=True)
        _plot_respuesta_mensual(driver, cap_obj)

def _on_tec_resp(change=None):
    w_cap_resp.layout.display = '' if w_tec_resp.value == 'Solar' else 'none'
    _actualizar_resp()

w_tec_resp.observe(_on_tec_resp,  names='value')
w_cap_resp.observe(_actualizar_resp, names='value')
display(widgets.VBox([widgets.HBox([w_tec_resp, w_cap_resp]), out_resp]))
_actualizar_resp(None)

# Bloque 10 — Escenarios de generacion sistémica

In [78]:
out_esc = widgets.Output()

ESCENARIOS = {
    'Base': {
        'descripcion': 'Dia normal con buen recurso. Referencia operativa.',
        'stats': {'Solar':'MODA_ALTA','Eolica':'MODA_ALTA','Hidro':'MODA_ALTA','Termica':'MODA_ALTA'},
    },
    'Conservador': {
        'descripcion': 'Dia nublado / poco viento. Renovables en moda baja, convencionales compensando.',
        'stats': {'Solar':'MODA_BAJA','Eolica':'MODA_BAJA','Hidro':'MODA_ALTA','Termica':'MODA_ALTA'},
    },
    'Pesimista': {
        'descripcion': 'Peor dia de recurso renovable. Sistema convencional respondiendo al maximo.',
        'stats': {'Solar':'MINIMA','Eolica':'MINIMA','Hidro':'MAXIMA','Termica':'MAXIMA'},
    },
    'Optimista': {
        'descripcion': 'Maxima generacion solar. Riesgo de vertimiento.',
        'stats': {'Solar':'MAXIMA','Eolica':'MODA_ALTA','Hidro':'MODA_ALTA','Termica':'MINIMA'},
    },
    'Sequia': {
        'descripcion': 'Hidro en minimos. Solar y Termica compensan la restriccion hidrica estacional.',
        'stats': {'Solar':'MAXIMA','Eolica':'MINIMA','Hidro':'MINIMA','Termica':'MAXIMA'},
    },
}

_ORDEN_TEC = ['Termica', 'Hidro', 'Eolica', 'Solar']

def _calcular_escenario(mes, escenario_key):
    esc   = ESCENARIOS[escenario_key]
    stats = esc['stats']
    cuartos_validos = []
    for h in range(24):
        for m in [15, 30, 45]:
            cuartos_validos.append(h * 100 + m)
        cuartos_validos.append((h + 1) * 100)
    result = pd.DataFrame(index=cuartos_validos)
    result.index.name = 'CUARTO'
    for tec in _ORDEN_TEC:
        if tec not in stats:
            result[tec] = 0.0
            continue
        col = stats[tec]
        sub = df_tec[(df_tec['MES']==mes)&(df_tec['TECNOLOGIA']==tec)].set_index('CUARTO')[col].sort_index()
        sub = sub.reindex(result.index)
        if tec == 'Solar':
            sub.loc[sub.index < CUARTO_AMANECER]  = 0.0
            sub.loc[sub.index > CUARTO_ATARDECER] = 0.0
        sub = sub.interpolate(method='linear', limit_direction='both')
        sub = sub.ffill().bfill().fillna(0.0)
        sub = sub.rolling(window=4, center=True, min_periods=1).mean()
        result[tec] = sub.clip(lower=0.0)
    result['TOTAL'] = result[_ORDEN_TEC].sum(axis=1)
    return result

def _plot_escenario(mes, escenario_key):
    esc    = ESCENARIOS[escenario_key]
    stats  = esc['stats']
    df_esc = _calcular_escenario(mes, escenario_key)
    if df_esc.empty:
        print(f'Sin datos para {MESES_NOMBRES[mes]} / {escenario_key}.'); return
    cuartos = df_esc.index.tolist()
    fig     = go.Figure()
    acum    = pd.Series(0.0, index=df_esc.index)
    for tec in _ORDEN_TEC:
        if tec not in df_esc.columns: continue
        valores  = df_esc[tec]
        col_stat = stats.get(tec, '-')
        color    = tec_color.get(tec, '#888888')
        fig.add_trace(go.Scatter(
            x=cuartos, y=(acum+valores).tolist(),
            customdata=valores.tolist(), mode='lines', line=dict(width=0),
            fillcolor=hex_to_rgba(color, opacity=0.55),
            fill='tonexty' if tec != _ORDEN_TEC[0] else 'tozeroy',
            name=f'{tec}  [{col_stat}]',
            hovertemplate=f'<b>{tec}</b> [{col_stat}]<br>%{{customdata:.1f}} MW<extra></extra>',
        ))
        acum = acum + valores
    fig.add_trace(go.Scatter(x=cuartos, y=df_esc['TOTAL'].tolist(),
        mode='lines', line=dict(color='#ffffff',width=2,dash='dot'),
        name='TOTAL',
        hovertemplate='<b>TOTAL: %{y:.1f} MW</b><extra></extra>'))
    _layout_oscuro(fig, y_title='Generacion (MW)', height=520)
    fig.update_xaxes(range=[15,2400], dtick=100)
    fig.update_layout(hovermode='x unified')
    _agregar_amanecer_atardecer(fig)
    fig.show()
    pico_total = float(df_esc['TOTAL'].max())
    hora_pico  = int(df_esc['TOTAL'].idxmax())
    items = [(SEV_OK, f'Pico sistemico: <b>{pico_total:.1f} MW</b> en C-{hora_pico}')]
    for tec in _ORDEN_TEC:
        if tec not in df_esc.columns: continue
        pico_tec = float(df_esc[tec].max())
        aporte   = (df_esc[tec].sum()/df_esc['TOTAL'].sum()*100 if df_esc['TOTAL'].sum()>0 else 0)
        col_stat = stats.get(tec, '-')
        items.append((SEV_OK,
            f'{tec} [{col_stat}]: pico <b>{pico_tec:.1f} MW</b>  |  aporte <b>{aporte:.1f}%</b>'))
    _resumen_critico(items, titulo=f'Escenario {escenario_key} · {MESES_NOMBRES[mes]}')

w_esc = widgets.Dropdown(
    options=list(ESCENARIOS.keys()), value='Conservador',
    description='Escenario:', style={'description_width':'90px'},
    layout=widgets.Layout(width='240px'),
)
w_mes_esc = widgets.Dropdown(
    options=[(MESES_NOMBRES[m], m) for m in MESES_DISP],
    value=MESES_DISP[0],
    description='Mes:', style={'description_width':'50px'},
    layout=widgets.Layout(width='200px'),
)
def _actualizar_esc(escenario, mes):
    with out_esc:
        out_esc.clear_output(wait=True)
        _plot_escenario(mes, escenario)
widgets.interactive_output(_actualizar_esc, {'escenario': w_esc, 'mes': w_mes_esc})
display(widgets.VBox([widgets.HBox([w_esc, w_mes_esc]), out_esc]))

# Dashboard v6 — HTML dinamico standalone

In [50]:
## ══════════════════════════════════════════════════════════════════════════════
# DASHBOARD v9 — Serializar datos + generar HTML standalone
# Tabs: Generacion · Rampas · Maximos y Rampas
# ══════════════════════════════════════════════════════════════════════════════
import json as _json

if 'Solar' not in FF or not FF['Solar']:
    FF['Solar'] = {mes: (_stats_por_cuarto(df_tec,{'TECNOLOGIA':'Solar','MES':mes})
                         .loc[lambda s:(s.index>=15)&(s.index<=2400)]['MODA_ALTA']
                         .div(_CAP_TEC['Solar']).fillna(0.0))
                   for mes in MESES_DISP
                   if not _stats_por_cuarto(df_tec,{'TECNOLOGIA':'Solar','MES':mes}).empty}

# ── TEC ───────────────────────────────────────────────────────────────────────
tec_data = {}
for _t in TECS_DISP:
    tec_data[_t] = {}
    for _m in MESES_DISP:
        _s = df_tec[(df_tec['TECNOLOGIA']==_t)&(df_tec['MES']==_m)][['CUARTO']+STATS_COLS]
        if not _s.empty:
            tec_data[_t][str(_m)] = {str(int(r.CUARTO)):{s:round(float(getattr(r,s)),2) for s in STATS_COLS} for r in _s.itertuples()}

ramp_data = {}
for _t in df_ramp_tec['TECNOLOGIA'].dropna().unique():
    ramp_data[str(_t)] = {}
    for _m in MESES_DISP:
        _s = df_ramp_tec[(df_ramp_tec['TECNOLOGIA']==_t)&(df_ramp_tec['MES']==_m)]
        if not _s.empty:
            ramp_data[str(_t)][str(_m)] = {str(int(r.DIA)):{'up':round(float(r.RAMP_UP_MAX),2),'down':round(float(r.RAMP_DOWN_MAX),2)} for r in _s.itertuples()}

ff_sol     = {str(m):{str(int(c)):round(float(v),6) for c,v in s.items()} for m,s in FF['Solar'].items()}
cap_tec_js = {k:round(float(v),2) for k,v in _CAP_TEC.items()}

# ── NODO ──────────────────────────────────────────────────────────────────────
nodo_data = {}
for _n in NODOS_DISP:
    nodo_data[_n] = {}
    for _t in NODO_TECS_DISP.get(_n,[]):
        _s = df_nodo[(df_nodo['NODO']==_n)&(df_nodo['TECNOLOGIA']==_t)]
        if _s.empty: continue
        nodo_data[_n][_t] = {}
        for _m in MESES_DISP:
            _sm = _s[_s['MES']==_m][['CUARTO']+STATS_COLS]
            if not _sm.empty:
                nodo_data[_n][_t][str(_m)] = {str(int(r.CUARTO)):{s:round(float(getattr(r,s)),2) for s in STATS_COLS} for r in _sm.itertuples()}

nodo_ramp = {}
for _n in NODOS_DISP:
    nodo_ramp[_n] = {}
    for _t in NODO_TECS_DISP.get(_n,[]):
        _s = df_ramp_nodo[(df_ramp_nodo['NODO']==_n)&(df_ramp_nodo['TECNOLOGIA']==_t)]
        if _s.empty: continue
        nodo_ramp[_n][_t] = {}
        for _m in MESES_DISP:
            _sm = _s[_s['MES']==_m]
            if not _sm.empty:
                nodo_ramp[_n][_t][str(_m)] = {str(int(r.DIA)):{'up':round(float(r.RAMP_UP_MAX),2),'down':round(float(r.RAMP_DOWN_MAX),2)} for r in _sm.itertuples()}

nodo_cap = {_n:{_t:round(float(_CAP_NODO_TEC.get((_n,_t),0)),2) for _t in NODO_TECS_DISP.get(_n,[])} for _n in NODOS_DISP}

# ── PLANTA ────────────────────────────────────────────────────────────────────
planta_data = {}
for _p in PLANTAS_DISP:
    planta_data[_p] = {}
    _s = df_planta[df_planta['PLANTA']==_p]
    for _m in MESES_DISP:
        _sm = _s[_s['MES']==_m][['CUARTO']+STATS_COLS]
        if not _sm.empty:
            planta_data[_p][str(_m)] = {str(int(r.CUARTO)):{s:round(float(getattr(r,s)),2) for s in STATS_COLS} for r in _sm.itertuples()}

planta_cap  = {_p:round(float(_CAP_PLANTA.get(_p,0) or 0),2) for _p in PLANTAS_DISP}
planta_tec  = {_p:str(_TEC_PLANTA.get(_p,'') or '')  for _p in PLANTAS_DISP}
planta_nodo = {_p:str(_NODO_PLANTA.get(_p,'') or '') for _p in PLANTAS_DISP}
planta_zona = {_p:str(_ZONA_PLANTA.get(_p,'') or '') for _p in PLANTAS_DISP}

# ── Flags de calidad de datos ──────────────────────────────────────────────────
_flags_js = {}
for _n in NODOS_DISP:
    for _t in NODO_TECS_DISP.get(_n, []):
        _cap = float(_CAP_NODO_TEC.get((_n, _t), 0) or 0)
        _sub = df_nodo[(df_nodo['NODO']==_n)&(df_nodo['TECNOLOGIA']==_t)]
        if _sub.empty: continue
        if _cap <= 0 and float(_sub['MAXIMA'].max()) > 0:
            _flags_js[f'{_n}\u00b7{_t}'] = 'cap0'
        elif _cap > 0 and float(_sub['MODA_ALTA'].max()) > _cap:
            _flags_js[f'{_n}\u00b7{_t}'] = 'over'
for _t in TECS_DISP:
    _cap = float(_CAP_TEC.get(_t, 0) or 0)
    _sub = df_tec[df_tec['TECNOLOGIA']==_t]
    if not _sub.empty and _cap > 0 and float(_sub['MODA_ALTA'].max()) > _cap:
        _flags_js[f'TEC\u00b7{_t}'] = 'over'
print(f'[DASHBOARD] series marcadas (flag): {len(_flags_js)}')

_meta = {
    'CAP_SOLAR':        round(float(CAP_SOLAR_ACTUAL),2),
    'UMBRAL_CRITICO':   UMBRAL_CRITICO,
    'UMBRAL_ATENCION':  UMBRAL_ATENCION,
    'CUARTO_AMANECER':  CUARTO_AMANECER,
    'CUARTO_ATARDECER': CUARTO_ATARDECER,
    'MESES':   MESES_DISP,
    'TECS':    TECS_DISP,
    'NODOS':   NODOS_DISP,
    'NTECS':   NODO_TECS_DISP,
    'STATS':   STATS_COLS,
    'VERANO':  sorted(list(MESES_VERANO)),
    'LLUVIOSO':sorted(list(MESES_LLUVIOSO)),
    'MNOM':    {str(k):v for k,v in MESES_NOMBRES.items()},
    'FLAGS':   _flags_js,
    'PLANTAS': PLANTAS_DISP,
    'PCAP':    planta_cap,
    'PTEC':    planta_tec,
    'PNODO':   planta_nodo,
    'PZONA':   planta_zona,
}

# ── Gen solar en el momento de la rampa máxima (hover) ───────────────────────
gen_sol_mes_js = {}
gen_sol_dia_js = {}
if not df_gen_solar_up.empty and not df_gen_solar_down.empty:
    for _t in TECS_DISP:
        gen_sol_mes_js[_t] = {}
        gen_sol_dia_js[_t] = {}
        for _m in MESES_DISP:
            _sub_r = df_ramp_tec[(df_ramp_tec['TECNOLOGIA']==_t)&(df_ramp_tec['MES']==_m)]
            _sg_up = df_gen_solar_up[(df_gen_solar_up['TECNOLOGIA']==_t)&(df_gen_solar_up['MES']==_m)]
            _sg_dn = df_gen_solar_down[(df_gen_solar_down['TECNOLOGIA']==_t)&(df_gen_solar_down['MES']==_m)]
            _up_val = _dn_val = 0.0
            if not _sub_r.empty and _sub_r['RAMP_UP_MAX'].max() > 0:
                _day = int(_sub_r.loc[_sub_r['RAMP_UP_MAX'].idxmax(), 'DIA'])
                _r = _sg_up[_sg_up['DIA']==_day]
                _up_val = round(float(_r['GEN_SOLAR'].values[0]), 2) if not _r.empty else 0.0
            if not _sub_r.empty and _sub_r['RAMP_DOWN_MAX'].min() < 0:
                _day = int(_sub_r.loc[_sub_r['RAMP_DOWN_MAX'].idxmin(), 'DIA'])
                _r = _sg_dn[_sg_dn['DIA']==_day]
                _dn_val = round(float(_r['GEN_SOLAR'].values[0]), 2) if not _r.empty else 0.0
            gen_sol_mes_js[_t][str(_m)] = {'up': _up_val, 'down': _dn_val}
            gen_sol_dia_js[_t][str(_m)] = {}
            for _row in _sg_up.itertuples():
                _d = str(int(_row.DIA))
                _r_dn = _sg_dn[_sg_dn['DIA']==_row.DIA]
                gen_sol_dia_js[_t][str(_m)][_d] = {
                    'up':   round(float(_row.GEN_SOLAR), 2),
                    'down': round(float(_r_dn['GEN_SOLAR'].values[0]), 2) if not _r_dn.empty else 0.0
                }
print(f'[GEN_SOL] tecnologias={list(gen_sol_mes_js.keys())}')

# ── Respuesta de rampas — datos históricos para el bloque 9 del dashboard ─────
# Replica exactamente _calcular_respuesta_mensual del notebook.
# Se guarda el valor histórico crudo (sin escalar). El factor se aplica en JS.
resp_js = {}
for _driver in ['Solar', 'Eolica']:
    _drv_rows = df_ramp_tec[df_ramp_tec['TECNOLOGIA'] == _driver]
    if _drv_rows.empty:
        continue
    resp_js[_driver] = {}
    for _m in MESES_DISP:
        _drv_mes = _drv_rows[_drv_rows['MES'] == _m].set_index('DIA')
        if _drv_mes.empty:
            continue
        _dia_peor  = int(_drv_mes['RAMP_DOWN_MAX'].abs().idxmax())
        _drv_hist  = round(float(_drv_mes.loc[_dia_peor, 'RAMP_DOWN_MAX']), 2)
        _hidro_mes = df_ramp_tec[(df_ramp_tec['TECNOLOGIA']=='Hidro')&(df_ramp_tec['MES']==_m)].set_index('DIA')
        _hidro_h   = round(float(_hidro_mes.loc[_dia_peor, 'RAMP_UP_MAX']), 2) if _dia_peor in _hidro_mes.index else 0.0
        _term_mes  = df_ramp_tec[(df_ramp_tec['TECNOLOGIA']=='Termica')&(df_ramp_tec['MES']==_m)].set_index('DIA')
        _term_h    = round(float(_term_mes.loc[_dia_peor, 'RAMP_UP_MAX']),  2) if _dia_peor in _term_mes.index  else 0.0
        resp_js[_driver][str(_m)] = {
            'dia':   _dia_peor,
            'drv':   _drv_hist,
            'hidro': _hidro_h,
            'term':  _term_h,
        }
print(f'[RESP] drivers serializados: {list(resp_js.keys())}')

_HTML = """<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>ETESA 2025</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
  *{box-sizing:border-box;margin:0;padding:0}
  body{background:#0e0e0e;color:#ccc;font-family:Inter,sans-serif;display:flex;flex-direction:column;height:100vh;overflow:hidden}
  header{background:#141414;padding:10px 24px;border-bottom:1px solid #252525;display:flex;align-items:center;gap:16px;flex-shrink:0}
  header h1{font-size:15px;font-weight:700;color:#fff}
  header small{font-size:11px;color:#555}
  .wrap{display:flex;flex:1;overflow:hidden}
  .sb{width:215px;background:#111;border-right:1px solid #1e1e1e;padding:14px 12px;display:flex;flex-direction:column;gap:12px;flex-shrink:0;overflow-y:auto}
  .sg{display:flex;flex-direction:column;gap:5px;transition:opacity .2s}
  .sg.dim{opacity:.15;pointer-events:none}
  .sg label{font-size:10px;color:#555;text-transform:uppercase;letter-spacing:.8px}
  .sg select{background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:4px;padding:5px 8px;font-size:12px;width:100%}
  .sg input[type=range]{width:100%;accent-color:#E8A838;margin-top:2px}
  .capnum{font-size:20px;font-weight:700;color:#E8A838;text-align:center}
  .caphint{font-size:10px;color:#333;text-align:center}
  .stat-checks{display:flex;flex-direction:column;gap:3px}
  .stat-row{display:flex;align-items:center;gap:6px;font-size:11px;color:#aaa;cursor:pointer;padding:2px 0}
  .stat-row input{accent-color:#E8A838;cursor:pointer}
  .dim-btns{display:flex;gap:6px}
  .dbtn{flex:1;padding:6px 0;font-size:11px;font-weight:600;background:#1c1c1c;color:#555;border:1px solid #2e2e2e;border-radius:4px;cursor:pointer;text-align:center;transition:all .15s}
  .dbtn.on{background:#252525;color:#fff;border-color:#E8A838}
  hr{border:none;border-top:1px solid #1a1a1a}
  .main{flex:1;display:flex;flex-direction:column;overflow:hidden;min-width:0}
  .tbar{display:flex;background:#111;border-bottom:1px solid #1e1e1e;padding:0 16px;flex-shrink:0}
  .tb{padding:11px 18px;font-size:12px;font-weight:600;color:#555;background:none;border:none;border-bottom:2px solid transparent;cursor:pointer;white-space:nowrap}
  .tb.on{color:#fff;border-bottom-color:#E8A838}
  .tab{display:none;flex:1;overflow-y:auto;padding:14px;flex-direction:column;gap:12px}
  .tab.on{display:flex}
  .row2{display:grid;grid-template-columns:1fr 1fr;gap:12px}
  .pbox{background:#141414;border-radius:6px;overflow:hidden;min-height:460px}
  ::-webkit-scrollbar{width:5px}
  ::-webkit-scrollbar-track{background:#0e0e0e}
  ::-webkit-scrollbar-thumb{background:#2a2a2a;border-radius:3px}
</style>
</head>
<body>
<header>
  <h1>&#128268; ETESA &mdash; Proyecciones 2025</h1>
  <small>Standalone &middot; sin servidor</small>
</header>
<div class="wrap">
  <div class="sb">
    <div class="sg">
      <label>Estacion</label>
      <select id="sel-estacion">
        <option value="todas">Todas</option>
        <option value="verano">Verano (Dic–Abr)</option>
        <option value="lluvioso">Lluvioso (May–Nov)</option>
      </select>
    </div>
    <hr>
    <div class="sg">
      <label>Dimension</label>
      <div class="dim-btns">
        <button class="dbtn on" id="dbtn-tec" onclick="setDim('tec')">Tecnologia</button>
        <button class="dbtn"    id="dbtn-nod" onclick="setDim('nod')">Nodo</button>
        <button class="dbtn"    id="dbtn-pla" onclick="setDim('pla')">Planta</button>
      </div>
    </div>
    <div class="sg" id="sg-tec">
      <label>Tecnologia</label>
      <select id="sel-tec"></select>
    </div>
    <div class="sg" id="sg-nod" style="display:none">
      <label>Nodo</label>
      <select id="sel-nod"></select>
    </div>
    <div class="sg" id="sg-tec-nod" style="display:none">
      <label>Tecnologia</label>
      <select id="sel-tec-nod"></select>
    </div>
    <div class="sg" id="sg-pla" style="display:none">
      <label>Planta</label>
      <select id="sel-pla"></select>
      <div id="pla-meta" style="font-size:10px;color:#777;line-height:1.7;margin-top:4px"></div>
    </div>
    <hr>
    <div class="sg" id="sg-meses">
      <label>Grupo 1</label>
      <div style="display:flex;gap:3px;margin-bottom:4px">
        <button onclick="setGrupo(1,'todos')"    style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Todos</button>
        <button onclick="setGrupo(1,'verano')"   style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Verano</button>
        <button onclick="setGrupo(1,'lluvioso')" style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Lluvioso</button>
      </div>
      <div class="stat-checks" id="mes-checks-1"></div>
      <div style="display:flex;gap:3px;margin-top:5px">
        <button id="g1-mes" onclick="setModo(1,'mes')" style="flex:1;font-size:10px;padding:3px 0;background:#E8A838;color:#000;border:none;border-radius:3px;cursor:pointer;font-weight:700">Por mes</button>
        <button id="g1-agr" onclick="setModo(1,'agr')" style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Agregado</button>
      </div>
    </div>
    <div class="sg" id="sg-meses2">
      <label style="display:flex;align-items:center;gap:6px">
        <input type="checkbox" id="cmp-on" onchange="toggleGrupo2()" style="accent-color:#E8A838">
        Grupo 2
      </label>
      <div id="g2-body" style="display:none">
        <div style="display:flex;gap:3px;margin-bottom:4px;margin-top:4px">
          <button onclick="setGrupo(2,'todos')"    style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Todos</button>
          <button onclick="setGrupo(2,'verano')"   style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Verano</button>
          <button onclick="setGrupo(2,'lluvioso')" style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Lluvioso</button>
        </div>
        <div class="stat-checks" id="mes-checks-2"></div>
        <div style="display:flex;gap:3px;margin-top:5px">
          <button id="g2-mes" onclick="setModo(2,'mes')" style="flex:1;font-size:10px;padding:3px 0;background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;border-radius:3px;cursor:pointer">Por mes</button>
          <button id="g2-agr" onclick="setModo(2,'agr')" style="flex:1;font-size:10px;padding:3px 0;background:#E8A838;color:#000;border:none;border-radius:3px;cursor:pointer;font-weight:700">Agregado</button>
        </div>
      </div>
    </div>
    <div class="sg" id="sg-stat">
      <label>Estadistico</label>
      <div class="stat-checks" id="stat-checks"></div>
    </div>
    <hr>
    <div class="sg" id="sg-ramp-tec">
      <label>Tecnologia rampa</label>
      <select id="sel-ramp-tec">
        <option value="Solar">Solar</option>
        <option value="Eolica">Eolica</option>
      </select>
    </div>
    <div class="sg" id="sg-cap">
      <label>Capacidad Solar</label>
      <div class="capnum" id="lbl-cap">&mdash; MW</div>
      <div class="caphint">arrastra para proyectar</div>
      <input type="range" id="sldr-cap">
    </div>
    <div class="sg" id="sg-mes">
      <label>Mes</label>
      <select id="sel-mes"></select>
    </div>
  </div>
  <div class="main">
    <div class="tbar">
      <button class="tb on" onclick="showTab(0)">Generacion</button>
      <button class="tb"    onclick="showTab(1)">Rampas</button>
      <button class="tb"    onclick="showTab(2)">M&aacute;ximos y Rampas</button>
    </div>
    <div id="t0" class="tab on">
      <div class="row2">
        <div class="pbox" id="p-perfil"></div>
        <div class="pbox" id="p-pico"></div>
      </div>
      <div style="background:#141414;border-radius:6px;padding:14px 18px;overflow-x:auto;margin-top:0">
        <div style="font-size:10px;color:#555;text-transform:uppercase;letter-spacing:.8px;margin-bottom:10px">Ventanas Horarias &mdash; Mediana</div>
        <div id="p-ventanas"></div>
      </div>
    </div>
    <div id="t1" class="tab">
      <div class="row2">
        <div class="pbox" id="p-ramp-mes"></div>
        <div class="pbox" id="p-ramp-dias"></div>
      </div>
    </div>
    <div id="t2" class="tab">
      <div class="row2">
        <div class="pbox" id="p-6c"></div>
        <div class="pbox" id="p-resp"></div>
      </div>
    </div>
  </div>
</div>
<script>
const TEC   = ##TEC##;
const RAMP  = ##RAMP##;
const FF    = ##FF##;
const CTEC  = ##CTEC##;
const NDATA = ##NDATA##;
const NRAMP = ##NRAMP##;
const NCAP  = ##NCAP##;
const PDATA = ##PDATA##;
const M     = ##META##;
const GEN_SOL_MES = ##GEN_SOL_MES##;
const GEN_SOL_DIA = ##GEN_SOL_DIA##;
const RESP  = ##RESP##;

const BG='#141414',PAPER='#1a1a1a',FC='#cccccc',GC='rgba(255,255,255,0.06)';
const TCOL={Solar:'#E8A838',Eolica:'#1D9E75',Hidro:'#378ADD',Termica:'#D85A30'};
const MCOL={1:'#4E79A7',2:'#F28E2B',3:'#E15759',4:'#76B7B2',5:'#59A14F',
            6:'#EDC948',7:'#B07AA1',8:'#FF9DA7',9:'#9C755F',10:'#BAB0AC',
            11:'#D4A6C8',12:'#86BCB6'};
const STAT_EST={
  MINIMA:    {dash:'dot',    w:1.5, op:0.50},
  MODA_BAJA: {dash:'dashdot',w:1.8, op:0.70},
  MEDIANA:   {dash:'dash',   w:1.8, op:0.65},
  MODA_ALTA: {dash:'solid',  w:2.5, op:1.00},
  MAXIMA:    {dash:'dot',    w:1.5, op:0.45},
};
const CFG={responsive:true,displaylogo:false,modeBarButtonsToRemove:['sendDataToCloud']};
const facCap=(v,c)=> (c && c>0) ? ((v||0)/c) : null;

let curDim='tec', curTab=0;

const gTec   = ()=>document.getElementById('sel-tec').value;
const gNod   = ()=>document.getElementById('sel-nod').value;
const gTecN  = ()=>document.getElementById('sel-tec-nod').value;
const gPla   = ()=>document.getElementById('sel-pla').value;
const gMes   = ()=>document.getElementById('sel-mes').value;
const gCap   = ()=>+document.getElementById('sldr-cap').value;
const gStats = ()=>[...document.querySelectorAll('#stat-checks input:checked')].map(i=>i.value);
const gEst   = ()=>document.getElementById('sel-estacion').value;

function getMeses(){
  const e=gEst();
  if(e==='verano')   return M.MESES.filter(m=>M.VERANO.includes(m));
  if(e==='lluvioso') return M.MESES.filter(m=>M.LLUVIOSO.includes(m));
  return M.MESES;
}

function gData(){
  if(curDim==='tec'){
    const t=gTec();
    return {stats:TEC[t]||{},ramp:RAMP[t]||{},cap:CTEC[t]||1,
            solStats:TEC['Solar']||{},solRamp:RAMP['Solar']||{},
            solCap:M.CAP_SOLAR,color:TCOL[t]||'#fff',label:t};
  } else if(curDim==='pla'){
    const p=gPla(), t=M.PTEC[p]||'';
    return {stats:PDATA[p]||{},ramp:{},cap:(M.PCAP[p]||1),
            solStats:TEC['Solar']||{},solRamp:RAMP['Solar']||{},
            solCap:M.CAP_SOLAR,color:TCOL[t]||'#fff',label:p};
  } else {
    const n=gNod(),t=gTecN();
    return {stats:NDATA[n]?.[t]||{},ramp:NRAMP[n]?.[t]||{},
            cap:(NCAP[n]?.[t] ?? 1),
            solStats:NDATA[n]?.Solar||{},solRamp:NRAMP[n]?.Solar||{},
            solCap:NCAP[n]?.Solar||1,color:TCOL[t]||'#fff',label:n+' \u00b7 '+t};
  }
}

const gRampTec = ()=>{const e=document.getElementById('sel-ramp-tec');return e?e.value:'Solar';};

function gRampData(){
  const tech=gRampTec();
  if(curDim!=='nod') return {ramp:RAMP[tech]||{}, cap:CTEC[tech]||1, tech};
  const n=gNod();
  return {ramp:NRAMP[n]?.[tech]||{}, cap:NCAP[n]?.[tech]||1, tech};
}

function updateRampTec(){
  const el=document.getElementById('sel-ramp-tec'); if(!el) return;
  const prev=el.value;
  let techs=['Solar','Eolica'];
  if(curDim==='nod'){
    const n=gNod();
    techs=techs.filter(t=>(NCAP[n]?.[t]||0)>0);
    if(!techs.length) techs=['Solar'];
  }
  el.innerHTML='';
  techs.forEach(t=>{const o=document.createElement('option');o.value=t;o.textContent=t;el.appendChild(o);});
  el.value = techs.includes(prev) ? prev : techs[0];
}

function lay(ytitle,height,xEx,yEx,lEx){
  return Object.assign({
    xaxis:Object.assign({showgrid:true,gridcolor:GC,zeroline:false},xEx||{}),
    yaxis:Object.assign({title:ytitle,showgrid:true,gridcolor:GC,zeroline:false},yEx||{}),
    hovermode:'x unified',
    plot_bgcolor:BG,paper_bgcolor:PAPER,
    font:{color:FC,family:'Inter,sans-serif'},
    legend:{orientation:'h',yanchor:'bottom',y:1.02,xanchor:'right',x:1,
            bgcolor:'rgba(0,0,0,0)',font:{size:11}},
    height:height||460,margin:{t:40,l:65,r:40,b:60},
  },lEx||{});
}
function sLines(){
  const vl=x=>({type:'line',x0:x,x1:x,y0:0,y1:1,yref:'paper',
                line:{color:'rgba(232,168,56,0.45)',width:1,dash:'dot'}});
  const an=(x,t)=>({x,xref:'x',y:1,yref:'paper',text:t,showarrow:false,
                    font:{size:10,color:'#E8A838'},yanchor:'bottom'});
  return {shapes:[vl(M.CUARTO_AMANECER),vl(M.CUARTO_ATARDECER)],
          annotations:[an(M.CUARTO_AMANECER,'Amanecer'),an(M.CUARTO_ATARDECER,'Atardecer')]};
}
function uShp(v){return v.map(({y,col,dash})=>({type:'line',x0:0,x1:1,xref:'paper',y0:y,y1:y,yref:'y',line:{color:col,width:1,dash}}));}
function uAnn(v){return v.map(({y,col,lbl})=>({x:1,xref:'paper',xanchor:'left',y,yref:'y',text:lbl,showarrow:false,font:{size:10,color:col}}));}
const sevCol=v=>Math.abs(v)>=M.UMBRAL_CRITICO?'#e74c3c':Math.abs(v)>=M.UMBRAL_ATENCION?'#f39c12':'#2ecc71';
function hexRgba(h,a){const r=parseInt(h.slice(1,3),16),g=parseInt(h.slice(3,5),16),b=parseInt(h.slice(5,7),16);return `rgba(${r},${g},${b},${a})`;}

// ── TAB 0: Generacion ─────────────────────────────────────────────────────────
const STAT_COL={MINIMA:'#9C755F',MODA_BAJA:'#4E79A7',MEDIANA:'#59A14F',MODA_ALTA:'#E8A838',MAXIMA:'#E15759'};
const _modoG = {1:'mes', 2:'agr'};

function setGrupo(g, preset){
  const id = 'mes-checks-'+g;
  document.querySelectorAll('#'+id+' input').forEach(inp=>{
    const m=+inp.value;
    if(preset==='todos')         inp.checked=true;
    else if(preset==='verano')   inp.checked=M.VERANO.includes(m);
    else if(preset==='lluvioso') inp.checked=M.LLUVIOSO.includes(m);
  });
  if(preset==='todos') setModo(g,'mes');
  else                 setModo(g,'agr');
  if(curTab===0) renderPerfil();
}

function setModo(g, modo){
  _modoG[g]=modo;
  const on='background:#E8A838;color:#000;border:none;font-weight:700';
  const off='background:#1c1c1c;color:#ccc;border:1px solid #2e2e2e;font-weight:normal';
  document.getElementById('g'+g+'-mes').style.cssText=document.getElementById('g'+g+'-mes').style.cssText.split(';').filter(s=>!s.includes('background')&&!s.includes('color')&&!s.includes('border')&&!s.includes('font')).join(';')+(modo==='mes'?on:off);
  document.getElementById('g'+g+'-agr').style.cssText=document.getElementById('g'+g+'-agr').style.cssText.split(';').filter(s=>!s.includes('background')&&!s.includes('color')&&!s.includes('border')&&!s.includes('font')).join(';')+(modo==='agr'?on:off);
  if(curTab===0) renderPerfil();
}

function toggleGrupo2(){
  const on=document.getElementById('cmp-on').checked;
  document.getElementById('g2-body').style.display=on?'':'none';
  if(curTab===0) renderPerfil();
}

function getMesesSel(g){
  return [...document.querySelectorAll('#mes-checks-'+g+' input:checked')].map(i=>+i.value);
}

function getGrupoLabel(meses){
  const vs=meses.filter(m=>M.VERANO.includes(m)), ll=meses.filter(m=>M.LLUVIOSO.includes(m));
  if(vs.length===M.VERANO.length && ll.length===0) return 'Verano';
  if(ll.length===M.LLUVIOSO.length && vs.length===0) return 'Lluvioso';
  if(meses.length===M.MESES.length) return 'Todos';
  if(meses.length===1) return M.MNOM[String(meses[0])];
  return meses.length+' meses';
}

function buildGrupoTraces(d, cap, meses, stats, modo, dashOverride){
  const traces=[];
  if(modo==='mes'){
    for(const m of meses){
      const md=d.stats[m]; if(!md) continue;
      const xs=Object.keys(md).map(Number).sort((a,b)=>a-b);
      for(const s of stats){
        const e=STAT_EST[s];
        traces.push({x:xs, y:xs.map(x=>facCap(md[String(x)]?.[s], cap)),
                     type:'scatter',mode:'lines',
                     line:{color:MCOL[m],width:e.w,dash:dashOverride||e.dash},
                     name:M.MNOM[String(m)]+' \u00b7 '+s,
                     hovertemplate:M.MNOM[String(m)]+' \u00b7 '+s+'<br>C-%{x}: <b>%{y:.1%}</b><extra></extra>'});
      }
    }
  } else {
    const label=getGrupoLabel(meses);
    for(const s of stats){
      const e=STAT_EST[s], col=STAT_COL[s]||'#fff';
      const allS=meses.map(m=>{
        const md=d.stats[m]; if(!md) return null;
        const xs=Object.keys(md).map(Number).sort((a,b)=>a-b);
        return {xs, ys:xs.map(x=>facCap(md[String(x)]?.[s], cap))};
      }).filter(Boolean);
      if(!allS.length) continue;
      const xs=allS[0].xs;
      const mean=xs.map((_,i)=>{
        const v=allS.map(a=>a.ys[i]||0);
        if(s==='MINIMA') return Math.min(...v);
        if(s==='MAXIMA') return Math.max(...v);
        return v.reduce((a,b)=>a+b,0)/v.length;
      });
      traces.push({x:xs, y:mean, type:'scatter', mode:'lines',
                   line:{color:col, width:e.w, dash:dashOverride||e.dash},
                   name:label+' \u00b7 '+s+' (n='+allS.length+')',
                   hovertemplate:label+' \u00b7 '+s+'<br>C-%{x}: <b>%{y:.1%}</b><extra></extra>'});
    }
  }
  return traces;
}

function renderPerfil(){
  const d=gData(), cap=d.cap, stats=gStats();
  const {shapes,annotations}=sLines();
  shapes.push({type:'line',x0:15,x1:2400,xref:'x',y0:1,y1:1,yref:'y',line:{color:'rgba(255,255,255,0.4)',width:1,dash:'dash'}});
  annotations.push({x:1,xref:'paper',xanchor:'left',y:1,yref:'y',text:'Cap 100%',showarrow:false,font:{size:10,color:'#dddddd'}});
  const _fk = curDim==='nod' ? (gNod()+'\u00b7'+gTecN()) : curDim==='pla' ? ('PLA\u00b7'+gPla()) : ('TEC\u00b7'+gTec());
  if(M.FLAGS && M.FLAGS[_fk]){
    annotations.push({x:0.5,xref:'paper',y:1.08,yref:'paper',xanchor:'center',
      text:(M.FLAGS[_fk]==='cap0'?'\u26a0 sin capacidad registrada (cap=0)':'\u26a0 generacion supera la capacidad (>100%)'),
      showarrow:false,font:{size:11,color:'#e74c3c'}});
  }
  const m1=getMesesSel(1), m2=getMesesSel(2);
  const cmpOn=document.getElementById('cmp-on').checked;
  if(!stats.length||!m1.length){Plotly.react('p-perfil',[],lay('Factor de generacion',460),CFG);return;}
  let traces=[...buildGrupoTraces(d,cap,m1,stats,_modoG[1],null)];
  if(cmpOn && m2.length) traces=[...traces,...buildGrupoTraces(d,cap,m2,stats,_modoG[2],'dash')];
  Plotly.react('p-perfil',traces,lay('Factor de generacion',460,{range:[15,2400],dtick:100},{tickformat:'.0%'},{shapes,annotations}),CFG);
}

function renderPico(){
  const d=gData(),cap=d.cap,col=d.color,meses=getMeses();
  const nombres=meses.map(m=>M.MNOM[String(m)]);
  if(!meses.length){Plotly.react('p-pico',[],lay('Factor de generacion',460),CFG);return;}
  const picos={};
  for(const s of M.STATS)
    picos[s]=meses.map(m=>(cap&&cap>0)?Math.max(...Object.values(d.stats[m]||{}).map(v=>(v[s]||0)/cap),0):null);
  const traces=[{x:[...nombres,...[...nombres].reverse()],
                 y:[...picos.MODA_ALTA,...picos.MODA_BAJA.slice().reverse()],
                 fill:'toself',fillcolor:hexRgba(col,0.12),
                 line:{color:'rgba(0,0,0,0)'},hoverinfo:'skip',showlegend:false}];
  for(const [s,e] of Object.entries(STAT_EST))
    traces.push({x:nombres,y:picos[s],type:'scatter',mode:'lines+markers',name:s,
                 line:{color:col,width:e.w,dash:e.dash},
                 marker:{size:6,opacity:e.op,line:{color:'#fff',width:0.8}},opacity:e.op,
                 hovertemplate:'%{x}<br>'+s+': <b>%{y:.1%}</b><extra></extra>'});
  const _allP=[].concat(...Object.values(picos)).filter(v=>v!=null);
  const _yMax=Math.max(1.05, _allP.length?Math.max(..._allP)*1.08:1.05);
  Plotly.react('p-pico',traces,
    lay('Factor de generacion (pico mensual)',460,{},{tickformat:'.0%',range:[0,_yMax]},
        {shapes:uShp([{y:1.0,col:'rgba(255,255,255,0.45)',dash:'dash'},{y:.95,col:'rgba(231,76,60,0.6)',dash:'dot'},{y:.80,col:'rgba(243,156,18,0.6)',dash:'dot'}]),
         annotations:uAnn([{y:1.0,col:'#dddddd',lbl:'Cap 100%'},{y:.95,col:'#e74c3c',lbl:'Critico 95%'},{y:.80,col:'#f39c12',lbl:'Atencion 80%'}])}),CFG);
}

// ── TAB 1: Rampas ─────────────────────────────────────────────────────────────
function renderRampasMes(){
  const rd=gRampData(), tech=rd.tech;
  const factor = tech==='Solar' ? gCap()/rd.cap : 1;
  const cap = tech==='Solar' ? gCap() : rd.cap;
  const meses=getMeses(), sd=rd.ramp;
  const labels=[],upA=[],dnA=[],upP=[],dnP=[],genSolUp=[],genSolDn=[];
  for(const m of meses){
    const days=sd[String(m)]; if(!days) continue;
    const ups=Object.values(days).map(v=>v.up),dns=Object.values(days).map(v=>v.down);
    upA.push(Math.max(...ups)); dnA.push(Math.min(...dns));
    upP.push(+(Math.max(...ups)*factor).toFixed(2)); dnP.push(+(Math.min(...dns)*factor).toFixed(2));
    labels.push(M.MNOM[String(m)]);
    genSolUp.push(+((GEN_SOL_MES[tech]?.[String(m)]?.up??0)*factor).toFixed(1));
    genSolDn.push(+((GEN_SOL_MES[tech]?.[String(m)]?.down??0)*factor).toFixed(1));
  }
  const UC=M.UMBRAL_CRITICO,UA=M.UMBRAL_ATENCION,capS=rd.cap.toFixed(0);
  const VS=new Set(M.VERANO);
  function evs(u){let v=0,i=0;for(const m of meses){const dd=sd[String(m)]||{};for(const dv of Object.values(dd)){const up=(dv.up||0)*factor,dn=Math.abs(dv.down||0)*factor;if(up>=u||dn>=u){VS.has(m)?v++:i++;}}}return{v,i};}
  const eA=evs(UA),eC=evs(UC);
  const showProy = tech==='Solar' && cap!==rd.cap;
  const hasGen = Object.keys(GEN_SOL_MES).length > 0;
  const traces=[
    {x:labels,y:upA,type:'scatter',mode:'lines+markers',line:{color:'rgba(180,180,180,0.5)',width:1.5,dash:'dot'},marker:{size:4},name:'Ramp-up '+tech+' ('+capS+' MW)',hovertemplate:'%{x}: <b>+%{y:.1f} MW</b><extra></extra>'},
    {x:labels,y:dnA,type:'scatter',mode:'lines+markers',line:{color:'rgba(180,180,180,0.5)',width:1.5,dash:'dot'},marker:{size:4},name:'Ramp-down '+tech+' ('+capS+' MW)',hovertemplate:'%{x}: <b>%{y:.1f} MW</b><extra></extra>'},
  ];
  if(showProy){
    traces.push({x:labels,y:upP,type:'scatter',mode:'lines+markers',line:{color:'#2ecc71',width:2.5},marker:{size:7,color:'#2ecc71',line:{color:'#fff',width:1}},name:'Ramp-up proy. ('+cap+' MW)',customdata:hasGen?genSolUp:undefined,hovertemplate:hasGen?'%{x}: <b>+%{y:.1f} MW</b><br>\u2600 Solar: <b>%{customdata:.1f} MW</b><extra></extra>':'%{x}: <b>+%{y:.1f} MW</b><extra></extra>'});
    traces.push({x:labels,y:dnP,type:'scatter',mode:'lines+markers',line:{color:'#e74c3c',width:2.5},marker:{size:7,color:'#e74c3c',line:{color:'#fff',width:1}},name:'Ramp-down proy. ('+cap+' MW)',customdata:hasGen?genSolDn:undefined,hovertemplate:hasGen?'%{x}: <b>%{y:.1f} MW</b><br>\u2600 Solar: <b>%{customdata:.1f} MW</b><extra></extra>':'%{x}: <b>%{y:.1f} MW</b><extra></extra>'});
  }
  Plotly.react('p-ramp-mes',traces,lay('Rampa MW / 15 min',460,{title:'Mes'},{},
    {shapes:uShp([{y:UA,col:'rgba(243,156,18,0.5)',dash:'dot'},{y:-UA,col:'rgba(243,156,18,0.5)',dash:'dot'},{y:UC,col:'rgba(231,76,60,0.7)',dash:'dash'},{y:-UC,col:'rgba(231,76,60,0.7)',dash:'dash'}]),
     annotations:uAnn([{y:UA,col:'#f39c12',lbl:'+'+UA+' MW \u00b7 V:'+eA.v+'/I:'+eA.i},{y:-UA,col:'#f39c12',lbl:'-'+UA+' MW \u00b7 V:'+eA.v+'/I:'+eA.i},{y:UC,col:'#e74c3c',lbl:'+'+UC+' MW \u00b7 V:'+eC.v+'/I:'+eC.i},{y:-UC,col:'#e74c3c',lbl:'-'+UC+' MW \u00b7 V:'+eC.v+'/I:'+eC.i}])}),CFG);
}

function renderRampasDias(){
  const rd=gRampData(), tech=rd.tech, mes=gMes();
  const factor = tech==='Solar' ? gCap()/rd.cap : 1;
  const dd=rd.ramp[mes]||{};
  const dias=Object.keys(dd).map(Number).sort((a,b)=>a-b);
  const up=dias.map(v=>dd[v].up*factor),dn=dias.map(v=>dd[v].down*factor);
  const cols=dias.map((_,i)=>sevCol(Math.max(Math.abs(up[i]),Math.abs(dn[i]))));
  const UC=M.UMBRAL_CRITICO,UA=M.UMBRAL_ATENCION;
  const hasGen = Object.keys(GEN_SOL_DIA).length > 0;
  const genSolUp = hasGen ? dias.map(d=>+((GEN_SOL_DIA[tech]?.[String(mes)]?.[String(d)]?.up??0)*factor).toFixed(1)) : [];
  const genSolDn = hasGen ? dias.map(d=>+((GEN_SOL_DIA[tech]?.[String(mes)]?.[String(d)]?.down??0)*factor).toFixed(1)) : [];
  Plotly.react('p-ramp-dias',[
    {x:dias,y:up,type:'bar',name:'Ramp-up '+tech,marker:{color:cols,opacity:.90},customdata:hasGen?genSolUp:undefined,hovertemplate:hasGen?'Dia %{x}: <b>+%{y:.1f} MW</b><br>\u2600 Solar: <b>%{customdata:.1f} MW</b><extra></extra>':'Dia %{x}: <b>+%{y:.1f} MW</b><extra></extra>'},
    {x:dias,y:dn,type:'bar',name:'Ramp-down '+tech,marker:{color:cols,opacity:.40},customdata:hasGen?genSolDn:undefined,hovertemplate:hasGen?'Dia %{x}: <b>%{y:.1f} MW</b><br>\u2600 Solar: <b>%{customdata:.1f} MW</b><extra></extra>':'Dia %{x}: <b>%{y:.1f} MW</b><extra></extra>'},
  ],lay('Rampa MW / 15 min',460,{title:'Dia del mes',dtick:1},{},
    {barmode:'overlay',bargap:.15,
     shapes:uShp([{y:UA,col:'rgba(243,156,18,0.6)',dash:'dot'},{y:-UA,col:'rgba(243,156,18,0.6)',dash:'dot'},{y:UC,col:'rgba(231,76,60,0.75)',dash:'dash'},{y:-UC,col:'rgba(231,76,60,0.75)',dash:'dash'}]),
     annotations:uAnn([{y:UA,col:'#f39c12',lbl:'+'+UA+' MW'},{y:-UA,col:'#f39c12',lbl:'-'+UA+' MW'},{y:UC,col:'#e74c3c',lbl:'+'+UC+' MW'},{y:-UC,col:'#e74c3c',lbl:'-'+UC+' MW'}])}),CFG);
}

// ── TAB 2: Maximos y Rampas ───────────────────────────────────────────────────
function render6c(){
  const d=gData(),mes=gMes(),cap=gCap(),factor=cap/d.solCap;
  const ss=d.solStats[mes]||{},xs=Object.keys(ss).map(Number).sort((a,b)=>a-b);
  if(!xs.length){Plotly.react('p-6c',[],lay('MODA_ALTA Solar (MW)',460),CFG);return;}
  const yA=xs.map(x=>ss[String(x)]?.MODA_ALTA||0);
  const yP=yA.map(v=>v*factor);
  const {shapes,annotations}=sLines();
  Plotly.react('p-6c',[
    {x:[...xs,...[...xs].reverse()],y:[...yP,...yA.slice().reverse()],fill:'toself',fillcolor:'rgba(232,168,56,0.15)',line:{color:'rgba(0,0,0,0)'},hoverinfo:'skip',name:'Incremento'},
    {x:xs,y:yA,type:'scatter',mode:'lines',line:{color:'rgba(180,180,180,0.7)',width:2,dash:'dot'},name:'Actual \u2014 '+d.solCap.toFixed(0)+' MW',hovertemplate:'C-%{x}: %{y:.1f} MW<extra></extra>'},
    {x:xs,y:yP,type:'scatter',mode:'lines',line:{color:'#E8A838',width:2.8},name:'Proyectado \u2014 '+cap+' MW',hovertemplate:'C-%{x}: %{y:.1f} MW<extra></extra>'},
  ],lay('MODA_ALTA Solar (MW)',460,{range:[15,2400]},{},{shapes,annotations}),CFG);
}

function renderRampaRespuesta(){
  // Replica exactamente _calcular_respuesta_mensual + _plot_respuesta_mensual
  // del Bloque 9 del notebook. factor = cap_obj / cap_actual (mismo para todos).
  const driver    = gRampTec();
  const capActual = CTEC[driver] || 1;
  const capObj    = driver === 'Solar' ? gCap() : capActual;
  const factor    = capObj / capActual;
  const meses     = getMeses();
  const UC=M.UMBRAL_CRITICO, UA=M.UMBRAL_ATENCION;
  const drvCol    = TCOL[driver] || '#E8A838';
  const capS      = capActual.toFixed(0);
  const capO      = capObj.toFixed(0);

  const labels=[],drvProy=[],hidroHist=[],hidroReq=[],termHist=[],termReq=[],brechas=[];

  for(const m of meses){
    const d = RESP[driver]?.[String(m)];
    if(!d) continue;
    const dp = +(Math.abs(d.drv) * factor).toFixed(2);
    const hh = d.hidro;
    const hr = +(hh * factor).toFixed(2);
    const th = d.term;
    const tr = +(th * factor).toFixed(2);
    const b  = +(dp - (hr + tr)).toFixed(2);
    labels.push(M.MNOM[String(m)]);
    drvProy.push(dp);
    hidroHist.push(hh);
    hidroReq.push(hr);
    termHist.push(th);
    termReq.push(tr);
    brechas.push(b);
  }

  if(!labels.length){Plotly.react('p-resp',[],lay('MW / 15 min',460),CFG);return;}

  const colBr = brechas.map(b => b > UC ? '#e74c3c' : b > 0 ? '#f39c12' : '#2ecc71');

  const traces = [
    {x:labels, y:drvProy, type:'bar',
     name:driver+' exige ('+capO+' MW)',
     marker:{color:drvCol, opacity:0.9},
     hovertemplate:'%{x}<br>'+driver+' exige: <b>+%{y:.1f} MW</b><extra></extra>'},
    {x:labels, y:hidroHist, type:'bar',
     name:'Hidro real ('+capS+' MW)',
     marker:{color:'rgba(55,138,221,0.50)', line:{color:'rgba(55,138,221,0.9)',width:1.5}},
     hovertemplate:'%{x}<br>Hidro real: <b>%{y:.1f} MW</b><extra></extra>'},
    {x:labels, y:hidroReq, type:'bar',
     name:'Hidro requerida ('+capO+' MW)',
     marker:{color:'rgba(55,138,221,0.90)'},
     hovertemplate:'%{x}<br>Hidro req: <b>%{y:.1f} MW</b><extra></extra>'},
    {x:labels, y:termHist, type:'bar',
     name:'T\u00e9rmica real ('+capS+' MW)',
     marker:{color:'rgba(216,90,48,0.50)', line:{color:'rgba(216,90,48,0.9)',width:1.5}},
     hovertemplate:'%{x}<br>T\u00e9rmica real: <b>%{y:.1f} MW</b><extra></extra>'},
    {x:labels, y:termReq, type:'bar',
     name:'T\u00e9rmica requerida ('+capO+' MW)',
     marker:{color:'rgba(216,90,48,0.90)'},
     hovertemplate:'%{x}<br>T\u00e9rmica req: <b>%{y:.1f} MW</b><extra></extra>'},
    {x:labels, y:brechas, type:'scatter', mode:'markers+text',
     name:'Brecha ('+driver+' \u2212 Respuesta)',
     marker:{symbol:'diamond', size:10, color:colBr, line:{color:'#fff',width:1}},
     text:brechas.map(b=>(b>=0?'+':'')+b.toFixed(0)),
     textposition:'top center',
     textfont:{size:9, color:colBr},
     hovertemplate:'%{x}<br>Brecha: <b>%{y:+.1f} MW</b><extra></extra>'},
  ];

  Plotly.react('p-resp', traces,
    lay('MW / 15 min (peor dia del mes)', 460, {title:'Mes'}, {},
      {barmode:'group', bargap:0.15,
       shapes: uShp([
         {y: UA, col:'rgba(243,156,18,0.5)', dash:'dot'},
         {y: UC, col:'rgba(231,76,60,0.6)',  dash:'dash'},
       ]),
       annotations: uAnn([
         {y: UA, col:'#f39c12', lbl:'Atenci\u00f3n '+UA+' MW'},
         {y: UC, col:'#e74c3c', lbl:'Cr\u00edtico '+UC+' MW'},
       ])}), CFG);
}

// ── TAB 0 ext: Ventanas Horarias ─────────────────────────────────────────────
const VTNAS=[
  {id:'MED_AM',label:'MED AM',v0:800, v1:1045},
  {id:'MAX',   label:'MAX',   v0:1100,v1:1445},
  {id:'MED_PM',label:'MED PM',v0:1500,v1:1645},
];
const VCOL={MED_AM:'#4E79A7',MAX:'#E8A838',MED_PM:'#E15759'};

function calcVentanas(){
  const meses=getMeses();
  const rows=[];
  const totMW={MED_AM:0,MAX:0,MED_PM:0};
  let totCap=0;

  if(curDim==='tec'){
    for(const tech of M.TECS){
      const cap=CTEC[tech]||0; if(!cap) continue;
      const row={ent:tech,cap};
      for(const v of VTNAS){
        let mv=[];
        for(const m of meses){
          const md=TEC[tech]?.[String(m)]; if(!md) continue;
          const w=Object.entries(md).filter(([c])=>+c>=v.v0&&+c<=v.v1).map(([,s])=>s.MEDIANA||0);
          if(w.length) mv.push(Math.max(...w));
        }
        const val=mv.length?mv.reduce((a,b)=>a+b,0)/mv.length:0;
        row[v.id+'_MW']=val; row[v.id+'_%']=cap>0?val/cap*100:0;
        totMW[v.id]+=val;
      }
      totCap+=cap; rows.push(row);
    }
  } else if(curDim==='pla'){
    for(const p of M.PLANTAS){
      const cap=M.PCAP[p]||0; if(!cap) continue;
      const row={ent:p,cap};
      for(const v of VTNAS){
        let mv=[];
        for(const m of meses){
          const md=PDATA[p]?.[String(m)]; if(!md) continue;
          const w=Object.entries(md).filter(([c])=>+c>=v.v0&&+c<=v.v1).map(([,s])=>s.MEDIANA||0);
          if(w.length) mv.push(Math.max(...w));
        }
        const val=mv.length?mv.reduce((a,b)=>a+b,0)/mv.length:0;
        row[v.id+'_MW']=val; row[v.id+'_%']=cap>0?val/cap*100:0;
        totMW[v.id]+=val;
      }
      totCap+=cap; rows.push(row);
    }
  } else {
    for(const nodo of M.NODOS){
      const cap=Object.values(NCAP[nodo]||{}).reduce((a,b)=>a+b,0); if(!cap) continue;
      const row={ent:nodo,cap};
      for(const v of VTNAS){
        let mv=[];
        for(const m of meses){
          const cs={};
          for(const t of (M.NTECS[nodo]||[])){
            const md=NDATA[nodo]?.[t]?.[String(m)]; if(!md) continue;
            for(const [c,s] of Object.entries(md)){
              if(+c>=v.v0&&+c<=v.v1) cs[c]=(cs[c]||0)+(s.MEDIANA||0);
            }
          }
          const w=Object.values(cs); if(w.length) mv.push(Math.max(...w));
        }
        const val=mv.length?mv.reduce((a,b)=>a+b,0)/mv.length:0;
        row[v.id+'_MW']=val; row[v.id+'_%']=cap>0?val/cap*100:0;
        totMW[v.id]+=val;
      }
      totCap+=cap; rows.push(row);
    }
  }

  const tot={ent:'\u2500\u2500 TOTAL \u2500\u2500',cap:totCap,isTotal:true};
  VTNAS.forEach(v=>{
    tot[v.id+'_MW']=totMW[v.id];
    tot[v.id+'_%']=totCap>0?totMW[v.id]/totCap*100:0;
  });
  rows.push(tot);
  return rows;
}

function renderVentanas(){
  const rows=calcVentanas();
  let h='<table style="width:100%;border-collapse:collapse;font-size:12px;color:#ccc">';
  h+='<thead>';
  h+='<tr style="background:#1c1c1c">';
  h+='<th style="text-align:left;padding:8px 14px;color:#777;font-weight:500">Entidad</th>';
  for(const v of VTNAS)
    h+=`<th colspan="2" style="text-align:center;padding:8px 10px;border-left:1px solid #2a2a2a;color:${VCOL[v.id]};font-weight:600">${v.label}</th>`;
  h+='</tr>';
  h+='<tr style="background:#181818;font-size:10px;color:#555"><th></th>';
  for(const v of VTNAS){
    h+='<th style="text-align:right;padding:3px 8px;border-left:1px solid #252525;font-weight:400">MW</th>';
    h+='<th style="text-align:right;padding:3px 8px;font-weight:400">%</th>';
  }
  h+='</tr></thead><tbody>';
  for(const row of rows){
    const bg=row.isTotal?'#1e1e1e':'transparent';
    const fw=row.isTotal?'700':'400';
    const bt=row.isTotal?'border-top:1px solid #333':'';
    h+=`<tr style="background:${bg};font-weight:${fw};${bt}">`;
    h+=`<td style="padding:6px 14px">${row.ent}</td>`;
    for(const v of VTNAS){
      const mw=row[v.id+'_MW']||0, pct=row[v.id+'_%']||0;
      const pc=pct>80?'#e74c3c':pct>60?'#f39c12':'#aaa';
      h+=`<td style="text-align:right;padding:6px 8px;border-left:1px solid #1e1e1e">${mw.toFixed(1)}</td>`;
      h+=`<td style="text-align:right;padding:6px 8px;color:${pc}">${pct.toFixed(1)}%</td>`;
    }
    h+='</tr>';
  }
  h+='</tbody></table>';
  document.getElementById('p-ventanas').innerHTML=h;
}

// ── navegacion ────────────────────────────────────────────────────────────────
const SB_ACT={
  0:['sg-meses','sg-meses2','sg-stat'],
  1:['sg-ramp-tec','sg-cap','sg-mes'],
  2:['sg-ramp-tec','sg-cap','sg-mes'],
};
const SB_OPT=['sg-meses','sg-meses2','sg-stat','sg-ramp-tec','sg-cap','sg-mes'];

function setDim(dim){
  curDim=dim;
  document.getElementById('dbtn-tec').classList.toggle('on',dim==='tec');
  document.getElementById('dbtn-nod').classList.toggle('on',dim==='nod');
  document.getElementById('dbtn-pla').classList.toggle('on',dim==='pla');
  document.getElementById('sg-tec').style.display     =dim==='tec'?'':'none';
  document.getElementById('sg-nod').style.display     =dim==='nod'?'':'none';
  document.getElementById('sg-tec-nod').style.display =dim==='nod'?'':'none';
  document.getElementById('sg-pla').style.display     =dim==='pla'?'':'none';
  if(dim==='pla') updatePlaMeta();
  updateSlider(); updateRampTec(); renderCur();
}

function updatePlaMeta(){
  const el=document.getElementById('pla-meta'); if(!el) return;
  const p=gPla(); if(!p){el.innerHTML='';return;}
  const t=M.PTEC[p]||'', col=TCOL[t]||'#fff';
  el.innerHTML=`<span style="color:${col};font-weight:600">${t}</span>`
    +` &middot; Nodo: <span style="color:#aaa">${M.PNODO[p]||'?'}</span>`
    +` &middot; Zona: <span style="color:#aaa">${M.PZONA[p]||'?'}</span>`
    +`<br>Cap. instalada: <span style="color:#E8A838">${(M.PCAP[p]||0).toFixed(2)} MW</span>`;
}

function updateSlider(){
  const d=gData();
  const s=document.getElementById('sldr-cap');
  const mn=Math.round(d.solCap),mx=Math.max(1600,Math.round(d.solCap*3));
  s.min=mn;s.max=mx;s.step=10;
  s.value=mn;
  document.getElementById('lbl-cap').textContent=gCap()+' MW';
}

function updateMesOpts(){
  const meses=getMeses();
  const el=document.getElementById('sel-mes');
  const prev=+el.value;
  el.innerHTML='';
  meses.forEach(m=>{const o=document.createElement('option');o.value=m;o.textContent=M.MNOM[String(m)];el.appendChild(o);});
  if(meses.includes(prev)) el.value=prev;
}

function showTab(i){
  curTab=i;
  document.querySelectorAll('.tab').forEach((t,j)=>t.classList.toggle('on',j===i));
  document.querySelectorAll('.tb') .forEach((b,j)=>b.classList.toggle('on',j===i));
  SB_OPT.forEach(id=>document.getElementById(id).classList.add('dim'));
  (SB_ACT[i]||[]).forEach(id=>document.getElementById(id).classList.remove('dim'));
  setTimeout(()=>window.dispatchEvent(new Event('resize')),60);
  renderCur();
}

function renderCur(){
  if(curTab===0){renderPerfil();renderPico();}
  if(curTab===1){renderRampasMes();renderRampasDias();}
  if(curTab===2){render6c();renderRampaRespuesta();}
  renderVentanas();
}

document.getElementById('sel-estacion').onchange=()=>{updateMesOpts();renderCur();};
document.getElementById('sel-tec').onchange=renderCur;
document.getElementById('sel-mes').onchange=renderCur;
document.getElementById('sel-nod').onchange=()=>{
  const n=gNod(),tecs=M.NTECS[n]||[];
  const el=document.getElementById('sel-tec-nod');
  el.innerHTML='';
  tecs.forEach(t=>{const o=document.createElement('option');o.value=t;o.textContent=t;el.appendChild(o);});
  updateSlider();updateRampTec();renderCur();
};
document.getElementById('sel-tec-nod').onchange=()=>{updateSlider();renderCur();};
document.getElementById('sel-pla').onchange=()=>{updatePlaMeta();updateSlider();renderCur();};
document.getElementById('sldr-cap').oninput=()=>{
  document.getElementById('lbl-cap').textContent=gCap()+' MW';
  renderCur();
};
document.getElementById('sel-ramp-tec').onchange=()=>{if(curTab===1||curTab===2)renderCur();};

(function boot(){
  const addOpts=(id,arr,vFn,tFn)=>{
    const el=document.getElementById(id);
    arr.forEach(v=>{const o=document.createElement('option');o.value=vFn(v);o.textContent=tFn(v);el.appendChild(o);});
  };
  addOpts('sel-tec', M.TECS,  v=>v,         v=>v);
  addOpts('sel-nod', M.NODOS, v=>v,         v=>v);
  addOpts('sel-pla', M.PLANTAS, v=>v,       v=>v);
  addOpts('sel-mes', M.MESES, v=>String(v), v=>M.MNOM[String(v)]);
  const n0=M.NODOS[0],t0=(M.NTECS[n0]||[])[0]||'';
  addOpts('sel-tec-nod',M.NTECS[n0]||[],v=>v,v=>v);
  M.STATS.forEach(s=>{
    const lbl=document.createElement('label');
    lbl.className='stat-row';
    lbl.innerHTML=`<input type="checkbox" value="${s}" ${s==='MODA_ALTA'?'checked':''}><span>${s}</span>`;
    lbl.querySelector('input').onchange=()=>{if(curTab===0)renderPerfil();};
    document.getElementById('stat-checks').appendChild(lbl);
  });
  [1,2].forEach(g=>{
    M.MESES.forEach(m=>{
      const lbl=document.createElement('label');
      lbl.className='stat-row';
      const chk=g===1?'checked':(M.VERANO.includes(m)?'checked':'');
      lbl.innerHTML=`<input type="checkbox" value="${m}" ${chk}><span>${M.MNOM[String(m)]}</span>`;
      lbl.querySelector('input').onchange=()=>{if(curTab===0)renderPerfil();};
      document.getElementById('mes-checks-'+g).appendChild(lbl);
    });
  });
  updatePlaMeta();
  updateSlider();
  updateRampTec();
  SB_OPT.forEach(id=>document.getElementById(id).classList.add('dim'));
  (SB_ACT[0]||[]).forEach(id=>document.getElementById(id).classList.remove('dim'));
  renderCur();
})();
</script>
</body>
</html>"""

# ── Generar HTML ──────────────────────────────────────────────────────────────
html_out = (_HTML
  .replace('##TEC##',         _json.dumps(tec_data,       ensure_ascii=False))
  .replace('##RAMP##',        _json.dumps(ramp_data,      ensure_ascii=False))
  .replace('##FF##',          _json.dumps(ff_sol,         ensure_ascii=False))
  .replace('##CTEC##',        _json.dumps(cap_tec_js,     ensure_ascii=False))
  .replace('##NDATA##',       _json.dumps(nodo_data,      ensure_ascii=False))
  .replace('##NRAMP##',       _json.dumps(nodo_ramp,      ensure_ascii=False))
  .replace('##NCAP##',        _json.dumps(nodo_cap,       ensure_ascii=False))
  .replace('##PDATA##',       _json.dumps(planta_data,    ensure_ascii=False))
  .replace('##META##',        _json.dumps(_meta,          ensure_ascii=False))
  .replace('##GEN_SOL_MES##', _json.dumps(gen_sol_mes_js, ensure_ascii=False))
  .replace('##GEN_SOL_DIA##', _json.dumps(gen_sol_dia_js, ensure_ascii=False))
  .replace('##RESP##',        _json.dumps(resp_js,        ensure_ascii=False))
)
with open('ETESA_DASHBOARD_2025.html','w',encoding='utf-8') as f:
    f.write(html_out)

size_mb = os.path.getsize('ETESA_DASHBOARD_2025.html')/1e6
print(f"ETESA_DASHBOARD_2025.html  -  {size_mb:.1f} MB")

[DASHBOARD] series marcadas (flag): 9
[GEN_SOL] tecnologias=['Eolica', 'Hidro', 'Solar', 'Termica']
[RESP] drivers serializados: ['Solar', 'Eolica']
ETESA_DASHBOARD_2025.html  -  23.3 MB


In [79]:
# ══════════════════════════════════════════════════════════════════════════════
# Bloque 6d-EXPORT — Ventanas Horarias → Excel (9 hojas: 3 dim × 3 estaciones)
# ══════════════════════════════════════════════════════════════════════════════
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT_6D = 'ventanas_horarias.xlsx'

_EST_EXPORT = {
    'Todas':    MESES_DISP,
    'Verano':   list(MESES_VERANO),
    'Lluvioso': list(MESES_LLUVIOSO),
}

_HDR_FILL = PatternFill('solid', fgColor='2C3E50')
_HDR_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=10)
_TOT_FILL = PatternFill('solid', fgColor='D6EAF8')
_TOT_FONT = Font(name='Arial', bold=True, size=10)
_DAT_FONT = Font(name='Arial', size=10)
_TTL_FONT = Font(name='Arial', bold=True, size=11)
_TOP_BDR  = Border(top=Side(style='medium', color='2C3E50'))
_CTR      = Alignment(horizontal='center', vertical='center', wrap_text=True)
_RGT      = Alignment(horizontal='right',  vertical='center')

with pd.ExcelWriter(OUTPUT_6D, engine='openpyxl') as writer:
    for dim_name, (df_base, col_ent, cap_d) in _FUENTES_6D.items():
        for est_name, meses in _EST_EXPORT.items():
            df_r = _calc_ventanas(df_base, col_ent, cap_d, meses)
            if df_r.empty:
                continue

            sheet = f'{dim_name[:5]}_{est_name}'          # e.g. Tecnol_Todas ≤31 chars
            df_r.reset_index().to_excel(writer, sheet_name=sheet,
                                        index=False, startrow=2)
            ws = writer.sheets[sheet]

            # ── Título (fila 1) ──────────────────────────────────────────────
            ws.merge_cells(start_row=1, start_column=1,
                           end_row=1, end_column=len(df_r.columns) + 1)
            title_cell = ws.cell(1, 1, f'Ventanas Horarias · {dim_name} · {est_name}')
            title_cell.font      = _TTL_FONT
            title_cell.alignment = _CTR

            # ── Cabecera (fila 3) ────────────────────────────────────────────
            for cell in ws[3]:
                cell.fill      = _HDR_FILL
                cell.font      = _HDR_FONT
                cell.alignment = _CTR

            # ── Filas de datos ───────────────────────────────────────────────
            n_data = len(df_r)
            for row in ws.iter_rows(min_row=4, max_row=3 + n_data):
                for i, cell in enumerate(row):
                    cell.font      = _DAT_FONT
                    cell.alignment = _RGT if i > 0 else Alignment(horizontal='left')

            # ── Fila TOTAL (última) ──────────────────────────────────────────
            for cell in ws[3 + n_data]:
                cell.fill   = _TOT_FILL
                cell.font   = _TOT_FONT
                cell.border = _TOP_BDR

            # ── Anchos de columna ────────────────────────────────────────────
            for col_cells in ws.columns:
                width = max(
                    len(str(c.value)) if c.value is not None else 0
                    for c in col_cells
                )
                ws.column_dimensions[get_column_letter(
                    col_cells[0].column)].width = max(width + 3, 10)

print(f'✓  {OUTPUT_6D}  →  {len(_FUENTES_6D) * len(_EST_EXPORT)} hojas')

✓  ventanas_horarias.xlsx  →  9 hojas
